# Thesis Results Analysis

Dieses Notebook enthält die zentrale Auswertung der finalen Machine-Unlearning-Läufe für die Bachelorarbeit.

Die Rohdaten der zwölf Gradient-Ascent-Läufe werden nur gelesen. Darauf aufbauend werden M0-Ausgangswerte, Lernraten-Sweep, Full Validation, Vergleiche der beiden Loss-Varianten, Kollateraleffekte, Matched-Forgetting, Referenzmodell-Auswertung und die verwendeten Ergebnisgrafiken erzeugt.

Die internen Bezeichnungen `full_window` und `masked` entsprechen dem sequenzweiten beziehungsweise token-selektiven Gradient Ascent.


## 1. Setup und Pfade

Zunächst wird Google Drive eingebunden und der zentrale Projektpfad definiert.

**Wichtig:** Falls dein Ordner anders heißt, musst du nur `ROOT` anpassen.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# Nur diesen Pfad anpassen, falls dein Drive-Ordner anders heißt.
# ------------------------------------------------------------------
ROOT = Path('/content/drive/MyDrive/overall_results')

RAW = ROOT / 'raw'
OUTPUT = ROOT / 'outputs'
TABLES = OUTPUT / 'tables'
FIGURES = OUTPUT / 'figures'

TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print('ROOT:   ', ROOT)
print('RAW:    ', RAW)
print('TABLES: ', TABLES)
print('FIGURES:', FIGURES)

assert ROOT.exists(), f'ROOT-Verzeichnis nicht gefunden: {ROOT}'
assert RAW.exists(), f'RAW-Verzeichnis nicht gefunden: {RAW}'

print('\nSetup erfolgreich.')


## 2. Definition der finalen Runs

Die folgende Zelle legt explizit fest, welche zwölf Unlearning-Runs in die finale Auswertung eingehen.

Dadurch ist im Notebook transparent dokumentiert, welche Lernraten und Methoden Bestandteil der Bachelorarbeits-Auswertung sind.

Erwartete Ordnerstruktur:

```text
raw/
├── full_window/
│   ├── lr_1e-06/
│   ├── lr_5e-06/
│   ├── lr_1e-05/
│   ├── lr_1.5e-05/
│   ├── lr_2e-05/
│   └── lr_3e-05/
└── masked/
    ├── lr_1e-06/
    ├── lr_5e-06/
    ├── lr_1e-05/
    ├── lr_1.5e-05/
    ├── lr_2e-05/
    └── lr_3e-05/
```


In [ ]:
LEARNING_RATES = [
    1e-6,
    5e-6,
    1e-5,
    1.5e-5,
    2e-5,
    3e-5,
]

FULL_WINDOW_RUNS = {
    1e-6: RAW / 'full_window' / 'lr_1e-06',
    5e-6: RAW / 'full_window' / 'lr_5e-06',
    1e-5: RAW / 'full_window' / 'lr_1e-05',
    1.5e-5: RAW / 'full_window' / 'lr_1.5e-05',
    2e-5: RAW / 'full_window' / 'lr_2e-05',
    3e-5: RAW / 'full_window' / 'lr_3e-05',
}

MASKED_RUNS = {
    1e-6: RAW / 'masked' / 'lr_1e-06',
    5e-6: RAW / 'masked' / 'lr_5e-06',
    1e-5: RAW / 'masked' / 'lr_1e-05',
    1.5e-5: RAW / 'masked' / 'lr_1.5e-05',
    2e-5: RAW / 'masked' / 'lr_2e-05',
    3e-5: RAW / 'masked' / 'lr_3e-05',
}

RUNS = {}
for lr, path in FULL_WINDOW_RUNS.items():
    RUNS[('full_window', lr)] = path
for lr, path in MASKED_RUNS.items():
    RUNS[('masked', lr)] = path

print(f'Definierte Runs: {len(RUNS)}')
for (method, lr), path in RUNS.items():
    print(f'{method:12s} | LR={lr:<8g} | {path}')


## 3. Validierung der Rohdaten

Bevor Daten kombiniert oder ausgewertet werden, werden alle zwölf Run-Ordner auf Vollständigkeit und Konsistenz geprüft.

Geprüft werden unter anderem:

- Vorhandensein aller Pflichtdateien
- 16 Gradient-Ascent-Updates
- Evaluationsschritte `0, 1, 2, 5, 10, 16`
- gleiche Benchmark-Version
- gleiches Forget-Set
- gleiches Mask-Set
- gleicher Seed
- gleicher Ausgangscheckpoint
- korrekte Lernrate und Loss-Variante
- identischer M0-Validation-Loss
- identische Step-0-Prompt-Evaluation zwischen allen Runs

Die Validierung verändert keine Rohdaten.


In [ ]:
REQUIRED_FILES = [
    'metadata.json',
    'evaluation_history.csv',
    'fact_history.csv',
    'group_history.csv',
    'utility_history.csv',
    'forget_set_loss_history.csv',
    'forget_set_window_loss_history.csv',
    'training_history.csv',
    'efficiency_summary.csv',
]

EXPECTED_EVAL_STEPS = [0, 1, 2, 5, 10, 16]
EXPECTED_TRAINING_STEPS = list(range(1, 17))
EXPECTED_MAX_STEPS = 16
EXPECTED_SEED = 123
EXPECTED_BENCHMARK_VERSION = '0.6-pakistan-islamabad-frozen-v1'
EXPECTED_FORGET_SET_ID = 'pakistan_islamabad_explicit_current_windows_final'
EXPECTED_MASK_SET_ID = 'pakistan_islamabad_masked_spans_final'


def load_metadata(run_dir):
    with open(run_dir / 'metadata.json', 'r', encoding='utf-8') as f:
        return json.load(f)

validation_rows = []
errors = []

for (method, lr), run_dir in RUNS.items():
    run_errors = []

    if not run_dir.exists():
        run_errors.append(f'Run-Verzeichnis fehlt: {run_dir}')
    else:
        for filename in REQUIRED_FILES:
            if not (run_dir / filename).exists():
                run_errors.append(f'Pflichtdatei fehlt: {filename}')

    core_files = [
        run_dir / 'metadata.json',
        run_dir / 'evaluation_history.csv',
        run_dir / 'utility_history.csv',
        run_dir / 'forget_set_loss_history.csv',
        run_dir / 'training_history.csv',
    ]

    if all(path.exists() for path in core_files):
        metadata = load_metadata(run_dir)
        evaluation = pd.read_csv(run_dir / 'evaluation_history.csv')
        utility = pd.read_csv(run_dir / 'utility_history.csv')
        forget = pd.read_csv(run_dir / 'forget_set_loss_history.csv')
        training = pd.read_csv(run_dir / 'training_history.csv')

        if metadata.get('benchmark_version') != EXPECTED_BENCHMARK_VERSION:
            run_errors.append(f"benchmark_version: {metadata.get('benchmark_version')}")
        if metadata.get('forget_set_id') != EXPECTED_FORGET_SET_ID:
            run_errors.append(f"forget_set_id: {metadata.get('forget_set_id')}")
        if metadata.get('mask_set_id') != EXPECTED_MASK_SET_ID:
            run_errors.append(f"mask_set_id: {metadata.get('mask_set_id')}")
        if metadata.get('seed') != EXPECTED_SEED:
            run_errors.append(f"seed: {metadata.get('seed')}")

        config = metadata.get('unlearning_config', {})
        if config.get('max_steps') != EXPECTED_MAX_STEPS:
            run_errors.append(f"max_steps: {config.get('max_steps')}")

        if not np.isclose(float(config.get('learning_rate')), float(lr), rtol=0, atol=1e-15):
            run_errors.append(
                f"learning_rate Metadata={config.get('learning_rate')} erwartet={lr}"
            )

        expected_loss_mode = 'masked' if method == 'masked' else 'full_window'
        actual_loss_mode = config.get('loss_mode')
        if actual_loss_mode != expected_loss_mode:
            run_errors.append(
                f'loss_mode={actual_loss_mode}, erwartet={expected_loss_mode}'
            )

        actual_training_steps = training['unlearning_step'].tolist()
        if actual_training_steps != EXPECTED_TRAINING_STEPS:
            run_errors.append(
                f'training_history unvollständig: {actual_training_steps}'
            )

        for name, df in [
            ('evaluation_history', evaluation),
            ('utility_history', utility),
            ('forget_set_loss_history', forget),
        ]:
            actual_steps = sorted(
                df['unlearning_step'].dropna().astype(int).unique().tolist()
            )
            if actual_steps != EXPECTED_EVAL_STEPS:
                run_errors.append(f'{name}: Schritte {actual_steps}')

        if 'forget_window_count' in forget.columns:
            window_counts = sorted(forget['forget_window_count'].dropna().unique().tolist())
            if window_counts != [16]:
                run_errors.append(f'forget_window_count={window_counts}')

        validation_rows.append({
            'method': method,
            'learning_rate': lr,
            'run_dir': str(run_dir),
            'run_id': metadata.get('run_name', metadata.get('run_id')),
            'source_checkpoint': metadata.get('source_checkpoint'),
            'benchmark_version': metadata.get('benchmark_version'),
            'forget_set_id': metadata.get('forget_set_id'),
            'mask_set_id': metadata.get('mask_set_id'),
            'seed': metadata.get('seed'),
            'm0_validation_loss': metadata.get('m0_validation_loss'),
            'status': 'OK' if not run_errors else 'FEHLER',
            'error_count': len(run_errors),
        })
    else:
        validation_rows.append({
            'method': method,
            'learning_rate': lr,
            'run_dir': str(run_dir),
            'run_id': None,
            'source_checkpoint': None,
            'benchmark_version': None,
            'forget_set_id': None,
            'mask_set_id': None,
            'seed': None,
            'm0_validation_loss': None,
            'status': 'FEHLER',
            'error_count': len(run_errors),
        })

    if run_errors:
        errors.append({'method': method, 'learning_rate': lr, 'errors': run_errors})

validation_df = pd.DataFrame(validation_rows)

display(validation_df[['method','learning_rate','status','error_count','run_id']])

if errors:
    print('\nVALIDIERUNGSFEHLER:')
    for item in errors:
        print(f"\n{item['method']} | LR={item['learning_rate']}")
        for error in item['errors']:
            print(' -', error)
else:
    print('\nAlle 12 Runs haben die strukturelle Validierung bestanden.')


In [ ]:
assert len(validation_df) == 12, f'Es wurden {len(validation_df)} statt 12 Runs gefunden.'
assert (validation_df['status'] == 'OK').all(), (
    'Mindestens ein Run hat die strukturelle Validierung nicht bestanden.'
)

source_checkpoints = validation_df['source_checkpoint'].dropna().unique()
print('Unterschiedliche source_checkpoints:', len(source_checkpoints))
for x in source_checkpoints:
    print(' -', x)
assert len(source_checkpoints) == 1, (
    'Die Runs verwenden unterschiedliche Ausgangscheckpoints.'
)

m0_losses = validation_df['m0_validation_loss'].dropna().astype(float)
print('\nM0 Validation Loss:')
print(m0_losses.describe())
assert np.isclose(m0_losses.max(), m0_losses.min(), rtol=0, atol=1e-12), (
    'M0 Validation Loss ist nicht über alle Runs identisch.'
)

print('\nÜbergreifende Metadata-Konsistenz: OK')


In [ ]:
m0_frames = []

for (method, lr), run_dir in RUNS.items():
    df = pd.read_csv(run_dir / 'evaluation_history.csv')
    m0 = df[df['unlearning_step'] == 0].copy()
    m0['analysis_method'] = method
    m0['analysis_learning_rate'] = lr
    m0_frames.append(m0)

all_m0 = pd.concat(m0_frames, ignore_index=True)

M0_KEYS = [
    'experiment_id',
    'group',
    'fact_id',
    'prompt_id',
    'target_type',
    'target',
]

m0_consistency = (
    all_m0
    .groupby(M0_KEYS, dropna=False)
    .agg(
        run_count=('analysis_method', 'size'),
        sequence_nll_min=('sequence_nll', 'min'),
        sequence_nll_max=('sequence_nll', 'max'),
        geo_prob_min=('geo_mean_probability', 'min'),
        geo_prob_max=('geo_mean_probability', 'max'),
        first_prob_min=('first_token_probability', 'min'),
        first_prob_max=('first_token_probability', 'max'),
        rank_min=('first_token_rank', 'min'),
        rank_max=('first_token_rank', 'max'),
    )
    .reset_index()
)

m0_consistency['sequence_nll_range'] = m0_consistency['sequence_nll_max'] - m0_consistency['sequence_nll_min']
m0_consistency['geo_prob_range'] = m0_consistency['geo_prob_max'] - m0_consistency['geo_prob_min']
m0_consistency['first_prob_range'] = m0_consistency['first_prob_max'] - m0_consistency['first_prob_min']
m0_consistency['rank_range'] = m0_consistency['rank_max'] - m0_consistency['rank_min']

print('Anzahl eindeutiger Step-0-Evaluationsfälle:', len(m0_consistency))
print('\nMaximale Abweichungen zwischen Runs:')
print(m0_consistency[['sequence_nll_range','geo_prob_range','first_prob_range','rank_range']].max())

assert (m0_consistency['run_count'] == 12).all(), (
    'Nicht jeder M0-Evaluationsfall ist in allen 12 Runs vorhanden.'
)
assert m0_consistency['sequence_nll_range'].max() < 1e-12
assert m0_consistency['geo_prob_range'].max() < 1e-12
assert m0_consistency['first_prob_range'].max() < 1e-12
assert m0_consistency['rank_range'].max() == 0

print('\nStep-0-Evaluation ist über alle 12 Runs identisch.')


## 4. Laden und Zusammenführen aller CSV-Dateien

Nachdem die zwölf Runs validiert wurden, werden die jeweiligen History-Dateien zu gemeinsamen DataFrames zusammengeführt.

Dabei werden zwei zusätzliche Analyse-Spalten ergänzt:

- `analysis_method`
- `analysis_learning_rate`

Die originalen CSV-Dateien werden dadurch nicht verändert.


In [ ]:
CSV_FILES = {
    'evaluation': 'evaluation_history.csv',
    'facts': 'fact_history.csv',
    'groups': 'group_history.csv',
    'utility': 'utility_history.csv',
    'forget_set': 'forget_set_loss_history.csv',
    'forget_set_windows': 'forget_set_window_loss_history.csv',
    'training': 'training_history.csv',
    'efficiency': 'efficiency_summary.csv',
}


def load_all_runs(filename):
    frames = []
    for (method, lr), run_dir in RUNS.items():
        path = run_dir / filename
        df = pd.read_csv(path)
        df['analysis_method'] = method
        df['analysis_learning_rate'] = lr
        df['analysis_run_dir'] = str(run_dir)
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


evaluation_all = load_all_runs(CSV_FILES['evaluation'])
facts_all = load_all_runs(CSV_FILES['facts'])
groups_all = load_all_runs(CSV_FILES['groups'])
utility_all = load_all_runs(CSV_FILES['utility'])
forget_set_all = load_all_runs(CSV_FILES['forget_set'])
forget_windows_all = load_all_runs(CSV_FILES['forget_set_windows'])
training_all = load_all_runs(CSV_FILES['training'])
efficiency_all = load_all_runs(CSV_FILES['efficiency'])

print('Geladene Zeilen:')
print(f'evaluation_all:     {len(evaluation_all):>6}')
print(f'facts_all:          {len(facts_all):>6}')
print(f'groups_all:         {len(groups_all):>6}')
print(f'utility_all:        {len(utility_all):>6}')
print(f'forget_set_all:     {len(forget_set_all):>6}')
print(f'forget_windows_all: {len(forget_windows_all):>6}')
print(f'training_all:       {len(training_all):>6}')
print(f'efficiency_all:     {len(efficiency_all):>6}')


In [ ]:
combined_tables = {
    'evaluation_all': evaluation_all,
    'facts_all': facts_all,
    'groups_all': groups_all,
    'utility_all': utility_all,
    'forget_set_all': forget_set_all,
    'forget_windows_all': forget_windows_all,
    'training_all': training_all,
    'efficiency_all': efficiency_all,
}

summary_rows = []
for name, df in combined_tables.items():
    summary_rows.append({
        'table': name,
        'rows': len(df),
        'methods': df['analysis_method'].nunique(),
        'learning_rates': df['analysis_learning_rate'].nunique(),
        'runs': df['run_id'].nunique() if 'run_id' in df.columns else np.nan,
    })

combined_summary = pd.DataFrame(summary_rows)
display(combined_summary)

assert (combined_summary['methods'] == 2).all()
assert (combined_summary['learning_rates'] == 6).all()

print('\nAlle kombinierten Tabellen enthalten beide Methoden und alle sechs Lernraten.')


## 5. M0-Ausgangszustand

Da die Validierung in Abschnitt 3 gezeigt hat, dass die Step-0-Evaluation
über alle zwölf Runs exakt identisch ist, wird für die weitere Analyse nur
ein gemeinsamer Ausgangszustand \(M_0\) verwendet.

Dieser Abschnitt erzeugt:

- die fünf primären Zielprompts von \(M_0\),
- die aggregierten Werte für `target_direct` und `target_inverse`,
- den bisherigen 50-Batch-Validation-Loss als **Monitoring-Wert**.

Die hier erzeugten CSV-Dateien werden unter `outputs/tables/` gespeichert.


In [ ]:
# ================================================================
# 5.1 Kanonische M0-Promptwerte
# ================================================================

m0_eval_all_runs = evaluation_all[
    evaluation_all["unlearning_step"] == 0
].copy()

# Nur die primären Zielantworten der eigentlichen Zielrelation.
m0_target_all_runs = m0_eval_all_runs[
    (m0_eval_all_runs["group"] == "target")
    & (m0_eval_all_runs["target_type"] == "primary")
].copy()

# Die Step-0-Werte sind über alle zwölf Runs identisch.
# Deshalb genau eine Zeile pro Prompt behalten.
m0_prompt_keys = [
    "experiment_id",
    "group",
    "fact_id",
    "prompt_id",
    "target_type",
    "target",
]

m0_target_prompts = (
    m0_target_all_runs
    .sort_values(["prompt_id"])
    .drop_duplicates(subset=m0_prompt_keys)
    .copy()
)

# Nur tatsächlich vorhandene, für die Darstellung sinnvolle Spalten wählen.
desired_prompt_columns = [
    "prompt_id",
    "prompt",
    "orientation",
    "prompt_type",
    "target",
    "sequence_nll",
    "geo_mean_probability",
    "first_token_probability",
    "first_token_rank",
    "target_token_count",
]

prompt_columns = [
    col for col in desired_prompt_columns
    if col in m0_target_prompts.columns
]

m0_target_prompts = m0_target_prompts[prompt_columns]

print("M0 – primäre Zielprompts:")
display(m0_target_prompts)

assert len(m0_target_prompts) == 5, (
    f"Es wurden {len(m0_target_prompts)} statt 5 primärer Target-Prompts gefunden."
)

m0_target_prompts.to_csv(
    TABLES / "m0_target_prompts.csv",
    index=False,
)

print(
    "\nGespeichert:",
    TABLES / "m0_target_prompts.csv"
)


In [ ]:
# ================================================================
# 5.2 M0 – direkte und inverse Zielrelation
# ================================================================

m0_target_groups_all_runs = groups_all[
    (groups_all["unlearning_step"] == 0)
    & (
        groups_all["analysis_group"].isin(
            ["target_direct", "target_inverse"]
        )
    )
].copy()

# Sicherstellen, dass die zwölf Kopien wirklich identisch sind.
group_check = (
    m0_target_groups_all_runs
    .groupby("analysis_group")
    .agg(
        nll_min=("mean_fact_nll", "min"),
        nll_max=("mean_fact_nll", "max"),
        prob_min=("mean_fact_geo_probability", "min"),
        prob_max=("mean_fact_geo_probability", "max"),
        rank_min=("median_fact_rank", "min"),
        rank_max=("median_fact_rank", "max"),
    )
)

assert np.allclose(
    group_check["nll_min"],
    group_check["nll_max"],
    rtol=0,
    atol=1e-12,
)

assert np.allclose(
    group_check["prob_min"],
    group_check["prob_max"],
    rtol=0,
    atol=1e-12,
)

assert (group_check["rank_min"] == group_check["rank_max"]).all()

# Eine kanonische Zeile pro Richtung.
m0_target_groups = (
    m0_target_groups_all_runs
    .sort_values(["analysis_group", "analysis_method", "analysis_learning_rate"])
    .drop_duplicates(subset=["analysis_group"])
    .copy()
)

m0_group_columns = [
    "analysis_group",
    "fact_count",
    "prompt_count",
    "mean_fact_nll",
    "mean_fact_geo_probability",
    "mean_fact_first_token_probability",
    "median_fact_rank",
]

m0_group_columns = [
    col for col in m0_group_columns
    if col in m0_target_groups.columns
]

m0_target_groups = m0_target_groups[m0_group_columns]

print("M0 – aggregierte Zielrelation:")
display(m0_target_groups)

m0_target_groups.to_csv(
    TABLES / "m0_target_groups.csv",
    index=False,
)

print(
    "\nGespeichert:",
    TABLES / "m0_target_groups.csv"
)


In [ ]:
# ================================================================
# 5.3 Bisheriger M0-Validation-Loss: 50-Batch-Monitoring
# ================================================================

m0_utility_rows = utility_all[
    utility_all["unlearning_step"] == 0
].copy()

m0_monitoring_losses = m0_utility_rows[
    "validation_loss"
].astype(float)

m0_monitoring_ppl = m0_utility_rows[
    "validation_perplexity"
].astype(float)

assert np.isclose(
    m0_monitoring_losses.max(),
    m0_monitoring_losses.min(),
    rtol=0,
    atol=1e-12,
)

assert np.isclose(
    m0_monitoring_ppl.max(),
    m0_monitoring_ppl.min(),
    rtol=0,
    atol=1e-12,
)

m0_monitoring_utility = pd.DataFrame([{
    "model_id": "M0",
    "validation_scope": "fixed_first_50_batches",
    "validation_batches": 50,
    "validation_batch_size": 4,
    "validation_context_length": 1024,
    "validation_stride": 1024,
    "validation_loss": float(m0_monitoring_losses.iloc[0]),
    "validation_perplexity": float(m0_monitoring_ppl.iloc[0]),
}])

print("M0 – bisheriger Validation-Monitoring-Wert:")
display(m0_monitoring_utility)

m0_monitoring_utility.to_csv(
    TABLES / "m0_validation_monitoring_50_batches.csv",
    index=False,
)

print(
    "\nHinweis: Dieser Wert wird später NICHT als finaler "
    "Full-Validation-Loss verwendet."
)


## 6. Gemeinsame Auswertung des Lernraten-Sweeps

Dieser Abschnitt kombiniert Full-Window- und Masked-GA über alle sechs
Lernraten und alle Evaluationsschritte.

Die zentralen Delta-Werte werden **explizit neu gegenüber Step 0 berechnet**,
auch wenn einzelne Rohdateien bereits Delta-Spalten enthalten.

Erzeugt werden unter anderem:

- mittlere NLL über alle fünf primären Target-Prompts,
- getrennte NLL für direkte und inverse Zielrichtung,
- Loss der 16 relationstragenden Tokenpositionen,
- Loss der übrigen Tokenpositionen derselben Forget-Fenster,
- Full-Window-Loss,
- 50-Batch-Validation-Loss als Monitoring-Wert.

Die vollständige Zeitreihe wird als `lr_sweep_history.csv` gespeichert.
Die Step-16-Zustände werden zusätzlich in `lr_sweep_step16.csv` exportiert.


In [ ]:
# ================================================================
# 6.1 Target-NLL über alle fünf primären Zielprompts
# ================================================================

target_primary = evaluation_all[
    (evaluation_all["group"] == "target")
    & (evaluation_all["target_type"] == "primary")
].copy()

target_all_history = (
    target_primary
    .groupby(
        [
            "analysis_method",
            "analysis_learning_rate",
            "unlearning_step",
        ],
        as_index=False,
    )
    .agg(
        target_prompt_count=("prompt_id", "nunique"),
        target_all_nll=("sequence_nll", "mean"),
    )
)

assert (target_all_history["target_prompt_count"] == 5).all()

# Baseline je Run-Konfiguration bestimmen.
target_baseline = (
    target_all_history[
        target_all_history["unlearning_step"] == 0
    ][
        [
            "analysis_method",
            "analysis_learning_rate",
            "target_all_nll",
        ]
    ]
    .rename(columns={"target_all_nll": "target_all_nll_m0"})
)

target_all_history = target_all_history.merge(
    target_baseline,
    on=["analysis_method", "analysis_learning_rate"],
    how="left",
    validate="many_to_one",
)

target_all_history["target_all_delta_nll"] = (
    target_all_history["target_all_nll"]
    - target_all_history["target_all_nll_m0"]
)

display(target_all_history.head(12))


In [ ]:
# ================================================================
# 6.2 Direkte und inverse Zielrichtung
# ================================================================

target_direction_history = groups_all[
    groups_all["analysis_group"].isin(
        ["target_direct", "target_inverse"]
    )
].copy()

target_direction_wide = (
    target_direction_history
    .pivot_table(
        index=[
            "analysis_method",
            "analysis_learning_rate",
            "unlearning_step",
        ],
        columns="analysis_group",
        values="mean_fact_nll",
        aggfunc="first",
    )
    .reset_index()
    .rename_axis(None, axis=1)
    .rename(columns={
        "target_direct": "target_direct_nll",
        "target_inverse": "target_inverse_nll",
    })
)

direction_baseline = (
    target_direction_wide[
        target_direction_wide["unlearning_step"] == 0
    ][
        [
            "analysis_method",
            "analysis_learning_rate",
            "target_direct_nll",
            "target_inverse_nll",
        ]
    ]
    .rename(columns={
        "target_direct_nll": "target_direct_nll_m0",
        "target_inverse_nll": "target_inverse_nll_m0",
    })
)

target_direction_wide = target_direction_wide.merge(
    direction_baseline,
    on=["analysis_method", "analysis_learning_rate"],
    how="left",
    validate="many_to_one",
)

target_direction_wide["target_direct_delta_nll"] = (
    target_direction_wide["target_direct_nll"]
    - target_direction_wide["target_direct_nll_m0"]
)

target_direction_wide["target_inverse_delta_nll"] = (
    target_direction_wide["target_inverse_nll"]
    - target_direction_wide["target_inverse_nll_m0"]
)

display(target_direction_wide.head(12))


In [ ]:
# ================================================================
# 6.3 Relation-, Context- und Full-Window-Loss
# ================================================================

forget_history = forget_set_all[
    [
        "analysis_method",
        "analysis_learning_rate",
        "unlearning_step",
        "full_window_nll",
        "relation_target_nll",
        "non_target_context_nll",
        "full_window_token_count",
        "relation_target_token_count",
        "non_target_context_token_count",
    ]
].copy()

forget_baseline = (
    forget_history[
        forget_history["unlearning_step"] == 0
    ][
        [
            "analysis_method",
            "analysis_learning_rate",
            "full_window_nll",
            "relation_target_nll",
            "non_target_context_nll",
        ]
    ]
    .rename(columns={
        "full_window_nll": "full_window_nll_m0",
        "relation_target_nll": "relation_target_nll_m0",
        "non_target_context_nll": "non_target_context_nll_m0",
    })
)

forget_history = forget_history.merge(
    forget_baseline,
    on=["analysis_method", "analysis_learning_rate"],
    how="left",
    validate="many_to_one",
)

forget_history["full_window_delta_nll"] = (
    forget_history["full_window_nll"]
    - forget_history["full_window_nll_m0"]
)

forget_history["relation_delta_nll"] = (
    forget_history["relation_target_nll"]
    - forget_history["relation_target_nll_m0"]
)

forget_history["context_delta_nll"] = (
    forget_history["non_target_context_nll"]
    - forget_history["non_target_context_nll_m0"]
)

# Nur deskriptive Differenz, kein eigener "Selectivity Score".
forget_history["relation_minus_context_delta"] = (
    forget_history["relation_delta_nll"]
    - forget_history["context_delta_nll"]
)

# Erwartete Tokenzahlen des finalen Forget Sets prüfen.
assert (
    forget_history["relation_target_token_count"] == 16
).all()

assert (
    forget_history["non_target_context_token_count"] == 16368
).all()

display(forget_history.head(12))


In [ ]:
# ================================================================
# 6.4 Validation-Loss-Monitoring über 50 Batches
# ================================================================

utility_monitoring_history = utility_all[
    [
        "analysis_method",
        "analysis_learning_rate",
        "unlearning_step",
        "validation_loss",
        "validation_perplexity",
    ]
].copy()

utility_baseline = (
    utility_monitoring_history[
        utility_monitoring_history["unlearning_step"] == 0
    ][
        [
            "analysis_method",
            "analysis_learning_rate",
            "validation_loss",
            "validation_perplexity",
        ]
    ]
    .rename(columns={
        "validation_loss": "validation_loss_m0_monitoring",
        "validation_perplexity": "validation_perplexity_m0_monitoring",
    })
)

utility_monitoring_history = utility_monitoring_history.merge(
    utility_baseline,
    on=["analysis_method", "analysis_learning_rate"],
    how="left",
    validate="many_to_one",
)

utility_monitoring_history["monitoring_validation_delta_loss"] = (
    utility_monitoring_history["validation_loss"]
    - utility_monitoring_history["validation_loss_m0_monitoring"]
)

utility_monitoring_history["monitoring_perplexity_ratio"] = (
    utility_monitoring_history["validation_perplexity"]
    / utility_monitoring_history["validation_perplexity_m0_monitoring"]
)

display(utility_monitoring_history.head(12))


In [ ]:
# ================================================================
# 6.5 Zentrale Sweep-Tabelle zusammenführen
# ================================================================

merge_keys = [
    "analysis_method",
    "analysis_learning_rate",
    "unlearning_step",
]

sweep_history = (
    target_all_history
    .merge(
        target_direction_wide,
        on=merge_keys,
        how="inner",
        validate="one_to_one",
    )
    .merge(
        forget_history,
        on=merge_keys,
        how="inner",
        validate="one_to_one",
    )
    .merge(
        utility_monitoring_history,
        on=merge_keys,
        how="inner",
        validate="one_to_one",
    )
)

# Übersichtliche Sortierung.
method_order = pd.CategoricalDtype(
    ["full_window", "masked"],
    ordered=True,
)

sweep_history["analysis_method"] = (
    sweep_history["analysis_method"].astype(method_order)
)

sweep_history = sweep_history.sort_values(
    [
        "analysis_method",
        "analysis_learning_rate",
        "unlearning_step",
    ]
).reset_index(drop=True)

# Für die weitere Analyse wichtige Spalten nach vorne.
sweep_columns = [
    "analysis_method",
    "analysis_learning_rate",
    "unlearning_step",
    "target_all_nll",
    "target_all_delta_nll",
    "target_direct_nll",
    "target_direct_delta_nll",
    "target_inverse_nll",
    "target_inverse_delta_nll",
    "relation_target_nll",
    "relation_delta_nll",
    "non_target_context_nll",
    "context_delta_nll",
    "relation_minus_context_delta",
    "full_window_nll",
    "full_window_delta_nll",
    "validation_loss",
    "monitoring_validation_delta_loss",
    "validation_perplexity",
    "monitoring_perplexity_ratio",
]

sweep_columns = [
    col for col in sweep_columns
    if col in sweep_history.columns
]

sweep_history_export = sweep_history[sweep_columns].copy()

assert len(sweep_history_export) == 72, (
    f"{len(sweep_history_export)} statt 72 Sweep-Zustände."
)

sweep_history_export.to_csv(
    TABLES / "lr_sweep_history.csv",
    index=False,
)

final_sweep = (
    sweep_history_export[
        sweep_history_export["unlearning_step"] == 16
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(final_sweep) == 12

final_sweep.to_csv(
    TABLES / "lr_sweep_step16.csv",
    index=False,
)

print("Step-16-Sweep:")
display(final_sweep)

print("\nGespeichert:")
print(" -", TABLES / "lr_sweep_history.csv")
print(" -", TABLES / "lr_sweep_step16.csv")


## 7. Finale Full-Validation-Evaluation

Die während der Unlearning-Runs gespeicherten Utility-Werte wurden nur auf
den ersten 50 Validation-Batches bestimmt. Sie bleiben als reproduzierbares
Monitoring für den Lernraten-Sweep erhalten.

Für die abschließende Utility-Bewertung werden nun zusätzlich folgende
Modellzustände über **sämtliche vollständigen Validation-Fenster** evaluiert:

- \(M_0\),
- alle zwölf Step-16-Unlearning-Zustände,
- \(M_{\mathrm{ref}}\).

Der Loss wird tokengewichtet über alle ausgewerteten Target-Tokens berechnet:

\[
L_{\mathrm{val}}^{\mathrm{full}}
=
\frac{\sum_i \mathrm{CE}_i}{N_{\mathrm{target\ tokens}}}.
\]

Dadurch wird auch der letzte eventuell kleinere DataLoader-Batch korrekt
gewichtet.

### Erwartete Checkpoints

Für jeden Unlearning-Run wird `ga_step_0016.pth` gesucht:

1. direkt im jeweiligen Run-Ordner oder
2. alternativ im Unterordner `checkpoints/`.

Die Pfade für \(M_0\), \(M_{\mathrm{ref}}\) und die Validation-Tokens sind
unten explizit definiert und können bei Bedarf angepasst werden.

Die Ergebnisse werden nach jedem Modellzustand in
`outputs/tables/full_validation_results.csv` gespeichert. Die Zelle ist
damit fortsetzbar, falls die Colab-Sitzung zwischenzeitlich unterbrochen wird.


In [ ]:
# ================================================================
# 7.1 Pfade, Modellarchitektur und Validation-Dataset
# ================================================================

import gc
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

MODEL_CONFIG = {
    "vocab_size": 50_257,
    "context_length": 1_024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

M0_CHECKPOINT = Path(
    "/content/drive/MyDrive/model_checkpoints/"
    "clean_baseline_v1/model_final_step_0046460.pth"
)

MREF_CHECKPOINT = Path(
    "/content/drive/MyDrive/model_checkpoints/"
    "reference_span_v1/model_final_step_0046460.pth"
)

VAL_TOKENS_PATH = Path(
    "/content/drive/MyDrive/simplewiki_val_tokens_exact.pt"
)

FULL_VAL_BATCH_SIZE = 4
FULL_VAL_CONTEXT_LENGTH = 1024
FULL_VAL_STRIDE = 1024

FULL_VALIDATION_OUTPUT = (
    TABLES / "full_validation_results.csv"
)

FULL_VALIDATION_VERSION = (
    "full_validation_v1_token_weighted_all_windows"
)


class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        d_in,
        d_out,
        context_length,
        dropout,
        num_heads,
        qkv_bias=False,
    ):
        super().__init__()

        if d_out % num_heads != 0:
            raise ValueError(
                "d_out muss durch num_heads teilbar sein."
            )

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(
            d_in, d_out, bias=qkv_bias
        )
        self.W_key = nn.Linear(
            d_in, d_out, bias=qkv_bias
        )
        self.W_value = nn.Linear(
            d_in, d_out, bias=qkv_bias
        )

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(
                    context_length,
                    context_length,
                ),
                diagonal=1,
            ),
        )

    def forward(self, x):
        batch_size, num_tokens, _ = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(
            batch_size,
            num_tokens,
            self.num_heads,
            self.head_dim,
        ).transpose(1, 2)

        queries = queries.view(
            batch_size,
            num_tokens,
            self.num_heads,
            self.head_dim,
        ).transpose(1, 2)

        values = values.view(
            batch_size,
            num_tokens,
            self.num_heads,
            self.head_dim,
        ).transpose(1, 2)

        attention_scores = (
            queries @ keys.transpose(2, 3)
        )

        causal_mask = (
            self.mask.bool()[
                :num_tokens,
                :num_tokens,
            ]
        )

        attention_scores.masked_fill_(
            causal_mask,
            -torch.inf,
        )

        attention_weights = torch.softmax(
            attention_scores
            / math.sqrt(self.head_dim),
            dim=-1,
        )

        attention_weights = self.dropout(
            attention_weights
        )

        context = (
            attention_weights @ values
        ).transpose(1, 2)

        context = context.reshape(
            batch_size,
            num_tokens,
            self.d_out,
        )

        return self.out_proj(context)


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()

        self.eps = 1e-5
        self.scale = nn.Parameter(
            torch.ones(emb_dim)
        )
        self.shift = nn.Parameter(
            torch.zeros(emb_dim)
        )

    def forward(self, x):
        mean = x.mean(
            dim=-1,
            keepdim=True,
        )

        variance = x.var(
            dim=-1,
            keepdim=True,
            unbiased=False,
        )

        normalized = (
            (x - mean)
            / torch.sqrt(
                variance + self.eps
            )
        )

        return (
            self.scale * normalized
            + self.shift
        )


class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (
            1.0
            + torch.tanh(
                math.sqrt(
                    2.0 / math.pi
                )
                * (
                    x
                    + 0.044715
                    * x.pow(3)
                )
            )
        )


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(
                cfg["emb_dim"],
                4 * cfg["emb_dim"],
            ),
            GELU(),
            nn.Linear(
                4 * cfg["emb_dim"],
                cfg["emb_dim"],
            ),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg[
                "context_length"
            ],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )

        self.ff = FeedForward(cfg)

        self.norm1 = LayerNorm(
            cfg["emb_dim"]
        )

        self.norm2 = LayerNorm(
            cfg["emb_dim"]
        )

        self.drop_shortcut = nn.Dropout(
            cfg["drop_rate"]
        )

    def forward(self, x):
        shortcut = x

        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x

        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(
            cfg["vocab_size"],
            cfg["emb_dim"],
        )

        self.pos_emb = nn.Embedding(
            cfg["context_length"],
            cfg["emb_dim"],
        )

        self.drop_emb = nn.Dropout(
            cfg["drop_rate"]
        )

        self.trf_blocks = nn.Sequential(
            *[
                TransformerBlock(cfg)
                for _ in range(
                    cfg["n_layers"]
                )
            ]
        )

        self.final_norm = LayerNorm(
            cfg["emb_dim"]
        )

        self.out_head = nn.Linear(
            cfg["emb_dim"],
            cfg["vocab_size"],
            bias=False,
        )

    def forward(self, in_idx):
        _, sequence_length = in_idx.shape

        if (
            sequence_length
            > self.pos_emb.num_embeddings
        ):
            raise ValueError(
                f"Sequenzlänge "
                f"{sequence_length} "
                f"überschreitet "
                f"Context Length "
                f"{self.pos_emb.num_embeddings}."
            )

        token_embeddings = (
            self.tok_emb(in_idx)
        )

        position_ids = torch.arange(
            sequence_length,
            device=in_idx.device,
        )

        position_embeddings = (
            self.pos_emb(position_ids)
        )

        x = (
            token_embeddings
            + position_embeddings
        )

        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)

        return self.out_head(x)


class CachedGPTDataset(Dataset):
    def __init__(
        self,
        token_tensor,
        max_length,
        stride,
    ):
        if token_tensor.ndim != 1:
            raise ValueError(
                "token_tensor muss "
                "eindimensional sein."
            )

        self.token_tensor = token_tensor
        self.max_length = max_length
        self.stride = stride

        self.num_windows = (
            1
            + (
                len(token_tensor)
                - max_length
                - 1
            )
            // stride
        )

    def __len__(self):
        return self.num_windows

    def __getitem__(self, index):
        start = index * self.stride

        input_ids = self.token_tensor[
            start : start + self.max_length
        ]

        target_ids = self.token_tensor[
            start + 1
            : start + self.max_length + 1
        ]

        return input_ids, target_ids


assert M0_CHECKPOINT.exists(), (
    f"M0 nicht gefunden: {M0_CHECKPOINT}"
)

assert MREF_CHECKPOINT.exists(), (
    f"M_ref nicht gefunden: {MREF_CHECKPOINT}"
)

assert VAL_TOKENS_PATH.exists(), (
    f"Validation-Tokens nicht gefunden: "
    f"{VAL_TOKENS_PATH}"
)

val_data = torch.load(
    VAL_TOKENS_PATH,
    map_location="cpu",
)

VAL_TOKENS = val_data["val_tokens"]

assert (
    val_data["context_length"]
    == FULL_VAL_CONTEXT_LENGTH
)

assert (
    val_data["stride"]
    == FULL_VAL_STRIDE
)

FULL_VAL_DATASET = CachedGPTDataset(
    VAL_TOKENS,
    max_length=FULL_VAL_CONTEXT_LENGTH,
    stride=FULL_VAL_STRIDE,
)

FULL_VAL_LOADER = DataLoader(
    FULL_VAL_DATASET,
    batch_size=FULL_VAL_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    pin_memory=(
        DEVICE.type == "cuda"
    ),
)

print("Device:", DEVICE)
print(
    "Validation-Tokens:",
    f"{len(VAL_TOKENS):,}",
)
print(
    "Validation-Fenster:",
    f"{len(FULL_VAL_DATASET):,}",
)
print(
    "Validation-Batches:",
    f"{len(FULL_VAL_LOADER):,}",
)
print(
    "Target-Tokens gesamt:",
    f"{len(FULL_VAL_DATASET) * FULL_VAL_CONTEXT_LENGTH:,}",
)


In [ ]:
# ================================================================
# 7.2 Verfügbare Checkpoints für diese Auswertungsrunde sammeln
# ================================================================

def find_step16_checkpoint_if_available(run_dir):
    """Sucht ga_step_0016.pth direkt im Run-Ordner oder unter checkpoints/.
    Gibt None zurück, wenn der Checkpoint aktuell nicht vorhanden ist.
    """
    candidates = [
        run_dir / "ga_step_0016.pth",
        run_dir / "checkpoints" / "ga_step_0016.pth",
    ]
    existing = [p for p in candidates if p.exists()]
    if len(existing) == 0:
        return None
    if len(existing) == 1:
        return existing[0]
    raise RuntimeError(f"Checkpoint doppelt vorhanden für {run_dir}: {existing}")

MODEL_STATES = []

if M0_CHECKPOINT.exists():
    MODEL_STATES.append({
        "model_id": "M0", "method": "baseline", "learning_rate": np.nan,
        "unlearning_step": 0, "checkpoint": M0_CHECKPOINT,
    })
else:
    print("NICHT VORHANDEN: M0", M0_CHECKPOINT)

for (method, lr), run_dir in RUNS.items():
    checkpoint = find_step16_checkpoint_if_available(run_dir)
    if checkpoint is None:
        print(f"NICHT VORHANDEN: {method} | LR={lr:g}")
        continue
    MODEL_STATES.append({
        "model_id": f"{method}_lr_{lr:g}_step16",
        "method": method,
        "learning_rate": float(lr),
        "unlearning_step": 16,
        "checkpoint": checkpoint,
    })

if MREF_CHECKPOINT.exists():
    MODEL_STATES.append({
        "model_id": "M_ref", "method": "reference", "learning_rate": np.nan,
        "unlearning_step": np.nan, "checkpoint": MREF_CHECKPOINT,
    })
else:
    print("NICHT VORHANDEN: M_ref", MREF_CHECKPOINT)

checkpoint_manifest = pd.DataFrame([
    {
        "model_id": s["model_id"], "method": s["method"],
        "learning_rate": s["learning_rate"], "unlearning_step": s["unlearning_step"],
        "checkpoint": str(s["checkpoint"]),
    }
    for s in MODEL_STATES
])

print(f"\nIn dieser Runde verfügbare Modellzustände: {len(MODEL_STATES)}")
display(checkpoint_manifest)


In [ ]:
# ================================================================
# 7.3 Checkpoint-Hilfsfunktionen
# ================================================================

def load_checkpoint_file(
    checkpoint_path,
    map_location="cpu",
):
    try:
        return torch.load(
            checkpoint_path,
            map_location=map_location,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            checkpoint_path,
            map_location=map_location,
        )


def looks_like_state_dict(candidate):
    return (
        isinstance(candidate, dict)
        and bool(candidate)
        and all(
            isinstance(k, str)
            and isinstance(v, torch.Tensor)
            for k, v
            in candidate.items()
        )
    )


def extract_model_state_dict(checkpoint):
    if looks_like_state_dict(checkpoint):
        return checkpoint

    if not isinstance(checkpoint, dict):
        raise TypeError(
            "Checkpoint ist weder State-Dict "
            "noch Dictionary."
        )

    for key in (
        "model_state_dict",
        "model_state",
        "state_dict",
        "model",
        "MODEL_STATE_DICT",
        "model_state_dict_cpu",
    ):
        candidate = checkpoint.get(key)

        if looks_like_state_dict(
            candidate
        ):
            return candidate

    raise KeyError(
        "Kein Modell-State-Dict gefunden. "
        f"Schlüssel: {list(checkpoint.keys())}"
    )


def strip_known_prefixes(state_dict):
    cleaned = dict(state_dict)

    for prefix in (
        "module.",
        "_orig_mod.",
        "model.",
    ):
        if (
            cleaned
            and all(
                key.startswith(prefix)
                for key in cleaned
            )
        ):
            cleaned = {
                key[len(prefix):]: value
                for key, value
                in cleaned.items()
            }

    return cleaned


def load_model_from_checkpoint(
    checkpoint_path,
    model_config,
    device,
):
    checkpoint = load_checkpoint_file(
        checkpoint_path,
        map_location="cpu",
    )

    state_dict = strip_known_prefixes(
        extract_model_state_dict(
            checkpoint
        )
    )

    model = GPTModel(model_config)

    model.load_state_dict(
        state_dict,
        strict=True,
    )

    model.to(device)
    model.eval()

    del checkpoint
    del state_dict

    return model


print(
    "Checkpoint-Hilfsfunktionen definiert."
)


In [ ]:
# ================================================================
# 7.4 Tokengewichteter Full-Validation-Loss
# ================================================================

@torch.inference_mode()
def evaluate_full_validation(
    model,
    data_loader,
    device,
):
    """
    Berechnet die mittlere negative Log-Likelihood
    über ALLE Target-Tokens des vollständigen Validation-Sets.

    Wichtig:
    - reduction='sum' pro Batch
    - anschließend Division durch die tatsächliche Zahl
      aller Target-Tokens
    - dadurch wird ein kleiner letzter Batch korrekt gewichtet
    """
    model.eval()

    total_nll = 0.0
    total_target_tokens = 0
    processed_windows = 0

    for input_batch, target_batch in data_loader:
        input_batch = input_batch.to(
            device,
            non_blocking=True,
        )

        target_batch = target_batch.to(
            device,
            non_blocking=True,
        )

        logits = model(input_batch)

        batch_nll = F.cross_entropy(
            logits.flatten(0, 1),
            target_batch.flatten(),
            reduction="sum",
        )

        total_nll += float(
            batch_nll.item()
        )

        total_target_tokens += int(
            target_batch.numel()
        )

        processed_windows += int(
            target_batch.shape[0]
        )

    if total_target_tokens == 0:
        raise RuntimeError(
            "Keine Validation-Tokens ausgewertet."
        )

    mean_nll = (
        total_nll
        / total_target_tokens
    )

    perplexity = math.exp(mean_nll)

    return {
        "full_validation_loss": mean_nll,
        "full_validation_perplexity": perplexity,
        "validation_window_count": processed_windows,
        "validation_target_token_count": total_target_tokens,
        "validation_batch_count": len(data_loader),
    }


print(
    "Full-Validation-Funktion definiert."
)


In [ ]:
# ================================================================
# 7.5 Alle 14 Modellzustände vollständig evaluieren
# ================================================================

# Fortsetzbares Logging:
# Existierende Ergebnisse dieser exakt benannten Version werden übernommen.
if FULL_VALIDATION_OUTPUT.exists():
    previous_results = pd.read_csv(
        FULL_VALIDATION_OUTPUT
    )

    if (
        "evaluation_version"
        in previous_results.columns
    ):
        previous_results = (
            previous_results[
                previous_results[
                    "evaluation_version"
                ]
                == FULL_VALIDATION_VERSION
            ]
            .copy()
        )
    else:
        previous_results = pd.DataFrame()

else:
    previous_results = pd.DataFrame()


if previous_results.empty:
    completed_ids = set()
    full_validation_rows = []
else:
    completed_ids = set(
        previous_results["model_id"]
        .astype(str)
        .tolist()
    )

    full_validation_rows = (
        previous_results
        .to_dict("records")
    )


print(
    "Bereits vollständig evaluiert:",
    sorted(completed_ids),
)


for state in MODEL_STATES:

    model_id = state["model_id"]

    if model_id in completed_ids:
        print(
            f"SKIP: {model_id} "
            "(bereits vorhanden)"
        )
        continue

    print(
        "\nEvaluierung:",
        model_id,
    )

    print(
        "Checkpoint:",
        state["checkpoint"],
    )

    model = load_model_from_checkpoint(
        state["checkpoint"],
        MODEL_CONFIG,
        DEVICE,
    )

    metrics = evaluate_full_validation(
        model,
        FULL_VAL_LOADER,
        DEVICE,
    )

    row = {
        "evaluation_version": (
            FULL_VALIDATION_VERSION
        ),
        "model_id": model_id,
        "method": state["method"],
        "learning_rate": (
            state["learning_rate"]
        ),
        "unlearning_step": (
            state["unlearning_step"]
        ),
        "checkpoint": str(
            state["checkpoint"]
        ),
        **metrics,
    }

    full_validation_rows.append(row)

    current_results = pd.DataFrame(
        full_validation_rows
    )

    current_results.to_csv(
        FULL_VALIDATION_OUTPUT,
        index=False,
    )

    print(
        "Full Validation Loss:",
        f"{metrics['full_validation_loss']:.6f}",
    )

    print(
        "Perplexity:",
        f"{metrics['full_validation_perplexity']:.6f}",
    )

    # GPU/CPU-Speicher zwischen Modellen freigeben.
    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


full_validation_results = pd.DataFrame(
    full_validation_rows
)

evaluated_model_count = full_validation_results["model_id"].nunique()

print(
    "\nAktuell vollständig ausgewertete Modellzustände:",
    evaluated_model_count,
    "/ 14",
)

if evaluated_model_count < 14:
    print(
        "Die Full Validation ist noch nicht vollständig. "
        "Fehlende Checkpoints können später ergänzt und die Zelle erneut ausgeführt werden."
    )
else:
    print("Alle 14 finalen Modellzustände wurden vollständig ausgewertet.")

# ---------------------------------------------------------------
# Delta und Perplexity-Verhältnis gegenüber M0
# ---------------------------------------------------------------

m0_full_row = (
    full_validation_results[
        full_validation_results[
            "model_id"
        ]
        == "M0"
    ]
    .iloc[0]
)

m0_full_loss = float(
    m0_full_row[
        "full_validation_loss"
    ]
)

m0_full_ppl = float(
    m0_full_row[
        "full_validation_perplexity"
    ]
)

full_validation_results[
    "delta_full_validation_loss"
] = (
    full_validation_results[
        "full_validation_loss"
    ]
    - m0_full_loss
)

full_validation_results[
    "full_validation_perplexity_ratio"
] = (
    full_validation_results[
        "full_validation_perplexity"
    ]
    / m0_full_ppl
)

# Sortierung für die Ausgabe.
method_sort = {
    "baseline": 0,
    "full_window": 1,
    "masked": 2,
    "reference": 3,
}

full_validation_results[
    "_method_order"
] = (
    full_validation_results[
        "method"
    ]
    .map(method_sort)
)

full_validation_results = (
    full_validation_results
    .sort_values(
        [
            "_method_order",
            "learning_rate",
        ],
        na_position="last",
    )
    .drop(
        columns="_method_order"
    )
    .reset_index(drop=True)
)

full_validation_results.to_csv(
    FULL_VALIDATION_OUTPUT,
    index=False,
)

print("\nFinale Full-Validation-Ergebnisse:")
display(
    full_validation_results[
        [
            "model_id",
            "method",
            "learning_rate",
            "full_validation_loss",
            "delta_full_validation_loss",
            "full_validation_perplexity",
            "full_validation_perplexity_ratio",
            "validation_window_count",
            "validation_target_token_count",
        ]
    ]
)

print(
    "\nGespeichert:",
    FULL_VALIDATION_OUTPUT,
)


### Interpretation der beiden Utility-Auswertungen

Nach Ausführung von Abschnitt 7 liegen zwei unterschiedliche Utility-Größen
vor:

- `validation_loss` aus `utility_history.csv`:
  fester Ausschnitt von 50 Batches, geeignet für den Verlauf des Sweeps;
- `full_validation_loss` aus `full_validation_results.csv`:
  sämtliche Validation-Fenster, vorgesehen für die abschließende
  Utility-Bewertung der finalen bzw. ausgewählten Modellzustände.

Die beiden Größen sollten im Ergebniskapitel nicht vermischt werden.


## 8. Kontrollierter Vergleich bei gleicher Lernrate

Der Same-LR-Vergleich hält Lernrate, 16 Updates, Forget-Set, Reihenfolge, Seed und Gradient-Clipping konstant. Er zeigt damit den Einfluss der Loss-Granularität unter gleicher Unlearning-Konfiguration. Gleiche Lernrate bedeutet ausdrücklich **nicht** gleiche erreichte Zielschwächung; dafür folgt später der Matched-Forgetting-Vergleich.


In [ ]:
# 8.1 Step-16-Vergleich Full-Window vs. Masked bei gleicher LR
same_lr_metrics = [
    "target_all_delta_nll", "target_direct_delta_nll", "target_inverse_delta_nll",
    "relation_delta_nll", "context_delta_nll", "relation_minus_context_delta",
    "full_window_delta_nll", "monitoring_validation_delta_loss",
]

full_step16 = final_sweep[final_sweep["analysis_method"] == "full_window"][
    ["analysis_learning_rate"] + same_lr_metrics
].copy()
masked_step16 = final_sweep[final_sweep["analysis_method"] == "masked"][
    ["analysis_learning_rate"] + same_lr_metrics
].copy()

full_step16 = full_step16.rename(columns={m: f"full_{m}" for m in same_lr_metrics})
masked_step16 = masked_step16.rename(columns={m: f"masked_{m}" for m in same_lr_metrics})

same_lr_step16 = full_step16.merge(
    masked_step16, on="analysis_learning_rate", how="inner", validate="one_to_one"
)
assert len(same_lr_step16) == 6

for metric in same_lr_metrics:
    same_lr_step16[f"masked_minus_full_{metric}"] = (
        same_lr_step16[f"masked_{metric}"] - same_lr_step16[f"full_{metric}"]
    )

same_lr_step16 = same_lr_step16.sort_values("analysis_learning_rate").reset_index(drop=True)

display(same_lr_step16[[
    "analysis_learning_rate",
    "full_target_all_delta_nll", "masked_target_all_delta_nll",
    "full_relation_delta_nll", "masked_relation_delta_nll",
    "full_context_delta_nll", "masked_context_delta_nll",
    "full_monitoring_validation_delta_loss", "masked_monitoring_validation_delta_loss",
]])

same_lr_step16.to_csv(TABLES / "same_lr_step16_comparison.csv", index=False)
print("Gespeichert:", TABLES / "same_lr_step16_comparison.csv")


In [ ]:
# 8.2 Same-LR-Vergleich über alle Evaluationsschritte
metrics = [
    "target_all_delta_nll", "relation_delta_nll", "context_delta_nll",
    "monitoring_validation_delta_loss",
]
keys = ["analysis_learning_rate", "unlearning_step"]

full_hist = sweep_history_export[sweep_history_export["analysis_method"] == "full_window"][keys + metrics].copy()
masked_hist = sweep_history_export[sweep_history_export["analysis_method"] == "masked"][keys + metrics].copy()
full_hist = full_hist.rename(columns={m: f"full_{m}" for m in metrics})
masked_hist = masked_hist.rename(columns={m: f"masked_{m}" for m in metrics})

same_lr_history = full_hist.merge(masked_hist, on=keys, how="inner", validate="one_to_one")
assert len(same_lr_history) == 36
for metric in metrics:
    same_lr_history[f"masked_minus_full_{metric}"] = (
        same_lr_history[f"masked_{metric}"] - same_lr_history[f"full_{metric}"]
    )

same_lr_history.to_csv(TABLES / "same_lr_history.csv", index=False)
print("Gespeichert:", TABLES / "same_lr_history.csv")


## 9. Tokenbezogene Selektivität innerhalb der Forget-Sequenzen

Verglichen wird die NLL-Änderung an den 16 annotierten relationstragenden Targetpositionen mit der NLL-Änderung an den übrigen 16.368 Targetpositionen derselben Forget-Fenster. `relation_minus_context_delta` wird nur als deskriptive Differenz verwendet und **nicht** als standardisierter Selectivity Score interpretiert.


In [ ]:
# 9.1 Relation vs. Context
token_selectivity_history = forget_history[[
    "analysis_method", "analysis_learning_rate", "unlearning_step",
    "relation_target_nll", "relation_delta_nll",
    "non_target_context_nll", "context_delta_nll",
    "relation_minus_context_delta", "full_window_nll", "full_window_delta_nll",
    "relation_target_token_count", "non_target_context_token_count",
]].copy().sort_values([
    "analysis_method", "analysis_learning_rate", "unlearning_step"
]).reset_index(drop=True)

assert len(token_selectivity_history) == 72
assert (token_selectivity_history["relation_target_token_count"] == 16).all()
assert (token_selectivity_history["non_target_context_token_count"] == 16368).all()

token_selectivity_step16 = token_selectivity_history[
    token_selectivity_history["unlearning_step"] == 16
].copy().reset_index(drop=True)

display(token_selectivity_step16[[
    "analysis_method", "analysis_learning_rate",
    "relation_delta_nll", "context_delta_nll", "relation_minus_context_delta",
]])

token_selectivity_history.to_csv(TABLES / "token_selectivity_history.csv", index=False)
token_selectivity_step16.to_csv(TABLES / "token_selectivity_step16.csv", index=False)


## 10. Kollateraleffekte auf Kontrollfakten

Die tokenbezogene Selektivität innerhalb der Forget-Sequenzen beantwortet nicht, ob anderes Faktenwissen betroffen ist. Deshalb werden `same_object`, `same_subject`, `same_window`, `same_relation_direct`, `same_relation_inverse` und `unrelated` separat gegenüber Step 0 ausgewertet. Die bestehenden `locality_*`-Variablennamen und Dateinamen bleiben aus Kompatibilitätsgründen erhalten.


In [ ]:
# 10.1 Locality-Historie aus group_history.csv
LOCALITY_GROUPS = [
    "same_object", "same_subject", "same_window",
    "same_relation_direct", "same_relation_inverse", "unrelated",
]

locality_raw = groups_all[groups_all["analysis_group"].isin(LOCALITY_GROUPS)].copy()
locality_baseline = locality_raw[locality_raw["unlearning_step"] == 0][[
    "analysis_method", "analysis_learning_rate", "analysis_group", "mean_fact_nll"
]].rename(columns={"mean_fact_nll": "mean_fact_nll_m0"})

locality_history = locality_raw.merge(
    locality_baseline,
    on=["analysis_method", "analysis_learning_rate", "analysis_group"],
    how="left", validate="many_to_one",
)
locality_history["delta_mean_fact_nll"] = (
    locality_history["mean_fact_nll"] - locality_history["mean_fact_nll_m0"]
)

locality_history = locality_history[[
    "analysis_method", "analysis_learning_rate", "unlearning_step", "analysis_group",
    "fact_count", "prompt_count", "mean_fact_nll", "mean_fact_nll_m0",
    "delta_mean_fact_nll", "median_fact_rank", "mean_fact_geo_probability",
    "mean_fact_first_token_probability",
]].copy()

expected_rows = 2 * 6 * 6 * len(LOCALITY_GROUPS)
assert len(locality_history) == expected_rows

locality_step16_wide = locality_history[locality_history["unlearning_step"] == 16].pivot_table(
    index=["analysis_method", "analysis_learning_rate"],
    columns="analysis_group", values="delta_mean_fact_nll", aggfunc="first"
).reset_index().rename_axis(None, axis=1)

display(locality_step16_wide)
locality_history.to_csv(TABLES / "locality_history.csv", index=False)
locality_step16_wide.to_csv(TABLES / "locality_step16.csv", index=False)


## 11. Rechenaufwand des Unlearnings

Für die Effizienz werden die eigentlichen GA-Updates betrachtet: Gesamtzeit, Median/Mittelwert pro Update, Updates pro Sekunde und Peak-GPU-Speicher. Die deutlich längere Evaluation bleibt davon getrennt. Der Anteil geclippter Updates wird nur als technische Diagnostik ergänzt.


In [ ]:
# 11.1 Effizienz pro Run
efficiency_analysis = efficiency_all[[
    "analysis_method", "analysis_learning_rate", "ga_update_count",
    "total_ga_update_time_seconds", "mean_ga_step_time_seconds",
    "median_ga_step_time_seconds", "ga_updates_per_second",
    "peak_gpu_memory_allocated_mb", "peak_gpu_memory_reserved_mb",
    "total_evaluation_compute_seconds", "total_run_wall_time_seconds",
]].copy()

clip_summary = training_all.groupby(
    ["analysis_method", "analysis_learning_rate"], as_index=False
).agg(
    training_step_count=("unlearning_step", "count"),
    clipped_step_count=("gradient_clipped", "sum"),
    median_pre_clip_gradient_norm=("pre_clip_gradient_norm", "median"),
)
clip_summary["gradient_clipped_fraction"] = (
    clip_summary["clipped_step_count"] / clip_summary["training_step_count"]
)

efficiency_analysis = efficiency_analysis.merge(
    clip_summary, on=["analysis_method", "analysis_learning_rate"],
    how="left", validate="one_to_one"
)
assert len(efficiency_analysis) == 12
assert (efficiency_analysis["ga_update_count"] == 16).all()

display(efficiency_analysis)
efficiency_analysis.to_csv(TABLES / "efficiency_analysis.csv", index=False)


In [ ]:
# 11.2 Effizienz nach Methode zusammenfassen
efficiency_by_method = efficiency_analysis.groupby("analysis_method", as_index=False).agg(
    runs=("analysis_learning_rate", "count"),
    mean_total_ga_seconds=("total_ga_update_time_seconds", "mean"),
    median_ga_step_seconds=("median_ga_step_time_seconds", "median"),
    mean_peak_allocated_mb=("peak_gpu_memory_allocated_mb", "mean"),
    mean_peak_reserved_mb=("peak_gpu_memory_reserved_mb", "mean"),
    mean_clipped_fraction=("gradient_clipped_fraction", "mean"),
    median_pre_clip_gradient_norm=("median_pre_clip_gradient_norm", "median"),
)

display(efficiency_by_method)
efficiency_by_method.to_csv(TABLES / "efficiency_by_method.csv", index=False)


## 12. Matched-Forgetting-Vergleich

Der Same-LR-Vergleich führt nicht zu gleicher Zielschwächung. Deshalb werden zusätzlich Full-Window- und Masked-Zustände mit möglichst ähnlicher Target-$\Delta$NLL gesucht. `target_all_delta_nll` beschreibt die Zielschwächung über die fünf primären Zielprompts; `relation_delta_nll` die Veränderung der 16 annotierten relationstragenden Positionen. Das Notebook exportiert zunächst transparente Matching-Kandidaten mit absoluter und relativer Abweichung.


In [ ]:
# 12.1 Zustände vorbereiten
match_states = sweep_history_export[sweep_history_export["unlearning_step"] > 0].copy()

locality_for_match = locality_history.pivot_table(
    index=["analysis_method", "analysis_learning_rate", "unlearning_step"],
    columns="analysis_group", values="delta_mean_fact_nll", aggfunc="first"
).reset_index().rename_axis(None, axis=1)

match_states = match_states.merge(
    locality_for_match,
    on=["analysis_method", "analysis_learning_rate", "unlearning_step"],
    how="left", validate="one_to_one"
)

full_states = match_states[match_states["analysis_method"] == "full_window"].copy()
masked_states = match_states[match_states["analysis_method"] == "masked"].copy()
assert len(full_states) == 30
assert len(masked_states) == 30
print("Matchbare Zustände:", len(full_states), "Full-Window +", len(masked_states), "Masked")


In [ ]:
# 12.2 Alle möglichen Paare für ein Matching-Kriterium
def build_matching_candidates(full_df, masked_df, match_metric):
    full = full_df.copy()
    masked = masked_df.copy()
    full["_join_key"] = 1
    masked["_join_key"] = 1

    pairs = full.merge(masked, on="_join_key", suffixes=("_full", "_masked")).drop(columns="_join_key")
    fcol = f"{match_metric}_full"
    mcol = f"{match_metric}_masked"

    pairs["match_abs_gap"] = (pairs[fcol] - pairs[mcol]).abs()
    mean_mag = (pairs[fcol].abs() + pairs[mcol].abs()) / 2.0
    pairs["match_relative_gap"] = np.where(mean_mag > 0, pairs["match_abs_gap"] / mean_mag, np.nan)
    pairs["matched_effect_mean"] = (pairs[fcol] + pairs[mcol]) / 2.0
    pairs["both_effects_positive"] = (pairs[fcol] > 0) & (pairs[mcol] > 0)
    pairs["match_metric"] = match_metric

    return pairs.sort_values(
        ["both_effects_positive", "match_relative_gap", "match_abs_gap"],
        ascending=[False, True, True]
    ).reset_index(drop=True)

target_match_candidates = build_matching_candidates(full_states, masked_states, "target_all_delta_nll")
relation_match_candidates = build_matching_candidates(full_states, masked_states, "relation_delta_nll")

target_match_candidates.to_csv(TABLES / "matched_forgetting_candidates_target.csv", index=False)
relation_match_candidates.to_csv(TABLES / "matched_forgetting_candidates_relation.csv", index=False)

print("Top-Kandidaten – Target-NLL-Matching:")
display(target_match_candidates[target_match_candidates["both_effects_positive"]][[
    "analysis_learning_rate_full", "unlearning_step_full", "target_all_delta_nll_full",
    "analysis_learning_rate_masked", "unlearning_step_masked", "target_all_delta_nll_masked",
    "match_abs_gap", "match_relative_gap", "matched_effect_mean",
]].head(15))

print("\nTop-Kandidaten – Relation-Token Matching:")
display(relation_match_candidates[relation_match_candidates["both_effects_positive"]][[
    "analysis_learning_rate_full", "unlearning_step_full", "relation_delta_nll_full",
    "analysis_learning_rate_masked", "unlearning_step_masked", "relation_delta_nll_masked",
    "match_abs_gap", "match_relative_gap", "matched_effect_mean",
]].head(15))


In [ ]:
# 12.3 Kompakte Kandidatenansicht mit Kollateraleffekten
COLLATERAL_METRICS = [
    "context_delta_nll", "monitoring_validation_delta_loss",
    "same_object", "same_subject", "same_window",
    "same_relation_direct", "same_relation_inverse", "unrelated",
]

def compact_match_view(candidates, match_metric, top_n=20):
    positive = candidates[candidates["both_effects_positive"]].copy()
    cols = [
        "analysis_learning_rate_full", "unlearning_step_full", f"{match_metric}_full",
        "analysis_learning_rate_masked", "unlearning_step_masked", f"{match_metric}_masked",
        "match_abs_gap", "match_relative_gap", "matched_effect_mean",
    ]
    for metric in COLLATERAL_METRICS:
        for suffix in ("_full", "_masked"):
            col = f"{metric}{suffix}"
            if col in positive.columns:
                cols.append(col)
    return positive[cols].head(top_n)

target_match_top20 = compact_match_view(target_match_candidates, "target_all_delta_nll")
relation_match_top20 = compact_match_view(relation_match_candidates, "relation_delta_nll")

target_match_top20.to_csv(TABLES / "matched_forgetting_top20_target.csv", index=False)
relation_match_top20.to_csv(TABLES / "matched_forgetting_top20_relation.csv", index=False)

display(target_match_top20)
display(relation_match_top20)


In [ ]:
import torch


def load_checkpoint_file(
    checkpoint_path,
    map_location="cpu",
):
    try:
        return torch.load(
            checkpoint_path,
            map_location=map_location,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            checkpoint_path,
            map_location=map_location,
        )


def looks_like_state_dict(candidate):
    return (
        isinstance(candidate, dict)
        and bool(candidate)
        and all(
            isinstance(k, str)
            and isinstance(v, torch.Tensor)
            for k, v in candidate.items()
        )
    )


def extract_model_state_dict(checkpoint):
    if looks_like_state_dict(checkpoint):
        return checkpoint

    if not isinstance(checkpoint, dict):
        raise TypeError(
            "Checkpoint ist weder State-Dict "
            "noch Dictionary."
        )

    for key in (
        "model_state_dict",
        "model_state",
        "state_dict",
        "model",
        "MODEL_STATE_DICT",
        "model_state_dict_cpu",
    ):
        candidate = checkpoint.get(key)

        if looks_like_state_dict(candidate):
            return candidate

    raise KeyError(
        "Kein Modell-State-Dict gefunden. "
        f"Schlüssel: {list(checkpoint.keys())}"
    )


def strip_known_prefixes(state_dict):
    cleaned = dict(state_dict)

    for prefix in (
        "module.",
        "_orig_mod.",
        "model.",
    ):
        if (
            cleaned
            and all(
                key.startswith(prefix)
                for key in cleaned
            )
        ):
            cleaned = {
                key[len(prefix):]: value
                for key, value
                in cleaned.items()
            }

    return cleaned


def load_model_from_checkpoint(
    checkpoint_path,
    model_config,
    device,
):
    checkpoint = load_checkpoint_file(
        checkpoint_path,
        map_location="cpu",
    )

    state_dict = strip_known_prefixes(
        extract_model_state_dict(checkpoint)
    )

    model = GPTModel(model_config)

    model.load_state_dict(
        state_dict,
        strict=True,
    )

    model.to(device)
    model.eval()

    del checkpoint
    del state_dict

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return model


print("Checkpoint-Ladefunktionen definiert.")

In [ ]:
from pathlib import Path
import gc
import pandas as pd
import torch


# ================================================================
# Zwei Matched-Forgetting-Zustände vollständig validieren
# ================================================================

MATCHED_CHECKPOINTS = [
    {
        "model_id": "full_window_lr_2e-05_step5",
        "method": "full_window",
        "learning_rate": 2e-5,
        "unlearning_step": 5,
        "checkpoint": Path("/content/drive/MyDrive/overall_results/raw/full_window/lr_2e-05/ga_step_0005.pth"
        ),
    },
    {
        "model_id": "masked_lr_1e-06_step10",
        "method": "masked",
        "learning_rate": 1e-6,
        "unlearning_step": 10,
        "checkpoint": Path(
            "/content/drive/MyDrive/overall_results/raw/masked/lr_1e-06/ga_step_0010.pth"
        ),
    },
]

MATCHED_OUTPUT = Path(
    "/content/drive/MyDrive/overall_results/"
    "outputs/tables/"
    "matched_full_validation_results.csv"
)


# ================================================================
# Sicherheitscheck
# ================================================================

for state in MATCHED_CHECKPOINTS:
    assert state["checkpoint"].exists(), (
        f"Checkpoint fehlt: {state['checkpoint']}"
    )

print("Beide Matched-Forgetting-Checkpoints gefunden.")


# ================================================================
# Bereits vorhandene Ergebnisse laden
# ================================================================

if MATCHED_OUTPUT.exists():
    matched_results = pd.read_csv(MATCHED_OUTPUT)
    rows = matched_results.to_dict("records")
    completed_ids = set(
        matched_results["model_id"].astype(str)
    )
else:
    rows = []
    completed_ids = set()


# ================================================================
# Full Validation
# ================================================================

for state in MATCHED_CHECKPOINTS:

    model_id = state["model_id"]

    if model_id in completed_ids:
        print(f"SKIP: {model_id} bereits ausgewertet")
        continue

    print("\nEvaluierung:", model_id)
    print("Checkpoint:", state["checkpoint"])

    model = load_model_from_checkpoint(
        state["checkpoint"],
        MODEL_CONFIG,
        DEVICE,
    )

    metrics = evaluate_full_validation(
        model,
        FULL_VAL_LOADER,
        DEVICE,
    )

    row = {
        "model_id": model_id,
        "method": state["method"],
        "learning_rate": state["learning_rate"],
        "unlearning_step": state["unlearning_step"],
        "checkpoint": str(state["checkpoint"]),
        **metrics,
    }

    rows.append(row)

    # Nach JEDEM Modell sofort speichern
    pd.DataFrame(rows).to_csv(
        MATCHED_OUTPUT,
        index=False,
    )

    print(
        "Full Validation Loss:",
        f"{metrics['full_validation_loss']:.6f}"
    )

    print(
        "Perplexity:",
        f"{metrics['full_validation_perplexity']:.6f}"
    )

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("\nMatched-Full-Validation abgeschlossen.")
print("Gespeichert:", MATCHED_OUTPUT)

In [ ]:
from pathlib import Path
import pandas as pd


MAIN_FULL_VAL = Path(
    "/content/drive/MyDrive/overall_results/"
    "outputs/tables/full_validation_results.csv"
)

MATCHED_OUTPUT = Path(
    "/content/drive/MyDrive/overall_results/"
    "outputs/tables/matched_full_validation_results.csv"
)


main_results = pd.read_csv(MAIN_FULL_VAL)
matched_results = pd.read_csv(MATCHED_OUTPUT)


# ================================================================
# M0-Referenz
# ================================================================

m0 = main_results[
    main_results["model_id"] == "M0"
].iloc[0]

m0_loss = float(
    m0["full_validation_loss"]
)

m0_ppl = float(
    m0["full_validation_perplexity"]
)


# ================================================================
# Deltas
# ================================================================

matched_results[
    "delta_full_validation_loss"
] = (
    matched_results["full_validation_loss"]
    - m0_loss
)

matched_results[
    "full_validation_perplexity_ratio"
] = (
    matched_results["full_validation_perplexity"]
    / m0_ppl
)

matched_results[
    "full_validation_perplexity_change_pct"
] = (
    (
        matched_results[
            "full_validation_perplexity_ratio"
        ]
        - 1
    )
    * 100
)


matched_results.to_csv(
    MATCHED_OUTPUT,
    index=False,
)

display(
    matched_results[
        [
            "model_id",
            "method",
            "learning_rate",
            "unlearning_step",
            "full_validation_loss",
            "delta_full_validation_loss",
            "full_validation_perplexity",
            "full_validation_perplexity_change_pct",
        ]
    ]
)


In [ ]:
# ================================================================
# M_ref auf dem Frozen Benchmark vollständig evaluieren
# ================================================================

from pathlib import Path
import json
import math
import gc

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import tiktoken


# ================================================================
# 1. Konfiguration
# ================================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

MREF_CHECKPOINT = Path(
    "/content/drive/MyDrive/model_checkpoints/"
    "reference_span_v1/model_final_step_0046460.pth"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/overall_results/"
    "outputs/tables"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EXPECTED_EXPERIMENT_ID = (
    "pakistan_capital_islamabad_final_v1"
)

EXPECTED_FACT_COUNT = 20
EXPECTED_PROMPT_COUNT = 40


# ================================================================
# 2. Frozen-Benchmark-Dateien suchen
# ================================================================
#
# Bevorzugt werden Snapshots eines FINALEN Unlearning-Runs,
# weil dadurch exakt derselbe Benchmark wie bei GA benutzt wird.
#
# Falls du facts.json / experiments.json separat im Drive
# gespeichert hast, kannst du deren Pfade hier ebenfalls ergänzen.
# ================================================================

FACT_CANDIDATES = [
    Path(
        "/content/drive/MyDrive/overall_results/raw/"
        "full_window/lr_1e-06/facts_snapshot.json"
    ),
    Path(
        "/content/drive/MyDrive/overall_results/raw/"
        "full_window/lr_1e-06/facts.json"
    ),
    Path(
        "/content/drive/MyDrive/machine_unlearning_experiment/"
        "benchmark/facts_pakistan-islamabad_final.json"
    ),
]

EXPERIMENT_CANDIDATES = [
    Path(
        "/content/drive/MyDrive/overall_results/raw/"
        "full_window/lr_1e-06/experiments_snapshot.json"
    ),
    Path(
        "/content/drive/MyDrive/overall_results/raw/"
        "full_window/lr_1e-06/experiments.json"
    ),
    Path(
        "/content/drive/MyDrive/machine_unlearning_experiment/"
        "benchmark/experiment_pakistan_capital_islamabad_final.json"
    ),
]


def first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    return None


FACTS_PATH = first_existing(
    FACT_CANDIDATES
)

EXPERIMENTS_PATH = first_existing(
    EXPERIMENT_CANDIDATES
)

assert FACTS_PATH is not None, (
    "Keine facts-Datei gefunden. "
    "Bitte FACT_CANDIDATES um den tatsächlichen "
    "Frozen-Benchmark-Pfad ergänzen."
)

assert EXPERIMENTS_PATH is not None, (
    "Keine experiments-Datei gefunden. "
    "Bitte EXPERIMENT_CANDIDATES um den tatsächlichen "
    "Frozen-Benchmark-Pfad ergänzen."
)

assert MREF_CHECKPOINT.exists(), (
    f"M_ref-Checkpoint fehlt: {MREF_CHECKPOINT}"
)

print("Facts:", FACTS_PATH)
print("Experiments:", EXPERIMENTS_PATH)
print("M_ref:", MREF_CHECKPOINT)


# ================================================================
# 3. Benchmark laden und FINALEN Stand prüfen
# ================================================================

with FACTS_PATH.open(
    "r",
    encoding="utf-8",
) as f:
    facts_document = json.load(f)

with EXPERIMENTS_PATH.open(
    "r",
    encoding="utf-8",
) as f:
    experiments_document = json.load(f)


assert "facts" in facts_document
assert "experiments" in experiments_document

facts = facts_document["facts"]
experiments = experiments_document["experiments"]

prompt_count = sum(
    len(fact["prompts"])
    for fact in facts
)

experiment_ids = [
    exp["experiment_id"]
    for exp in experiments
]

print("\nBenchmark-Sanity-Check:")
print("Fakten:", len(facts))
print("Prompts:", prompt_count)
print("Experimente:", experiment_ids)

assert len(facts) == EXPECTED_FACT_COUNT, (
    f"Erwartet {EXPECTED_FACT_COUNT} Fakten, "
    f"gefunden: {len(facts)}. "
    "Möglicherweise wurde ein alter Benchmark geladen."
)

assert prompt_count == EXPECTED_PROMPT_COUNT, (
    f"Erwartet {EXPECTED_PROMPT_COUNT} Prompts, "
    f"gefunden: {prompt_count}. "
    "Möglicherweise wurde ein alter Benchmark geladen."
)

assert EXPECTED_EXPERIMENT_ID in experiment_ids, (
    f"Finales Experiment "
    f"{EXPECTED_EXPERIMENT_ID!r} nicht gefunden. "
    f"Gefunden: {experiment_ids}"
)

experiment = next(
    exp
    for exp in experiments
    if exp["experiment_id"]
    == EXPECTED_EXPERIMENT_ID
)

fact_map = {
    fact["fact_id"]: fact
    for fact in facts
}

print("\nFinaler Frozen Benchmark bestätigt.")


# ================================================================
# 4. Tokenizer
# ================================================================

tokenizer = tiktoken.get_encoding(
    "gpt2"
)

assert (
    tokenizer.n_vocab
    == MODEL_CONFIG["vocab_size"]
)


# ================================================================
# 5. Exakt dieselbe Prompt-Scoring-Logik wie bei GA
# ================================================================

@torch.no_grad()
def score_target_sequence(
    model,
    tokenizer,
    prompt,
    target,
):
    was_training = model.training
    model.eval()

    try:
        device = next(
            model.parameters()
        ).device

        prompt_ids = tokenizer.encode(
            prompt
        )

        target_ids = tokenizer.encode(
            target
        )

        combined_ids = tokenizer.encode(
            prompt + target
        )

        if (
            not prompt_ids
            or not target_ids
        ):
            raise ValueError(
                "Prompt und Target dürfen "
                "nicht leer sein."
            )

        if (
            combined_ids
            != prompt_ids + target_ids
        ):
            raise ValueError(
                "Instabile Token-Grenze: "
                f"Prompt={prompt!r}, "
                f"Target={target!r}"
            )

        input_ids = torch.tensor(
            combined_ids[:-1],
            dtype=torch.long,
            device=device,
        ).unsqueeze(0)

        logits = model(
            input_ids
        )

        log_probs = F.log_softmax(
            logits,
            dim=-1,
        )

        start_position = (
            len(prompt_ids) - 1
        )

        target_length = len(
            target_ids
        )

        relevant = log_probs[
            0,
            start_position:
            start_position
            + target_length,
            :
        ]

        target_tensor = torch.tensor(
            target_ids,
            dtype=torch.long,
            device=device,
        )

        token_log_probs = (
            relevant
            .gather(
                1,
                target_tensor.unsqueeze(1),
            )
            .squeeze(1)
        )

        token_probabilities = (
            token_log_probs.exp()
        )

        sequence_nll = float(
            -token_log_probs
            .mean()
            .item()
        )

        total_sequence_nll = float(
            -token_log_probs
            .sum()
            .item()
        )

        first_token_logits = logits[
            0,
            start_position,
        ]

        first_id = target_ids[0]

        first_logit = (
            first_token_logits[
                first_id
            ]
        )

        first_rank = int(
            (
                first_token_logits
                > first_logit
            )
            .sum()
            .item()
        ) + 1

        first_probability = float(
            torch.softmax(
                first_token_logits,
                dim=-1,
            )[first_id].item()
        )

        return {
            "sequence_nll":
                sequence_nll,

            "total_sequence_nll":
                total_sequence_nll,

            "geo_mean_probability":
                math.exp(
                    -sequence_nll
                ),

            "first_token_probability":
                first_probability,

            "first_token_rank":
                first_rank,

            "target_token_count":
                target_length,

            "target_token_ids":
                target_ids,

            "target_tokens": [
                tokenizer.decode([x])
                for x in target_ids
            ],

            "token_log_probs":
                token_log_probs
                .cpu()
                .tolist(),

            "token_probabilities":
                token_probabilities
                .cpu()
                .tolist(),
        }

    finally:
        model.train(
            was_training
        )


# ================================================================
# 6. Komplette Frozen-Benchmark-Evaluation
# ================================================================

def evaluate_reference_model(
    model,
    tokenizer,
):
    rows = []

    for (
        group_name,
        fact_ids,
    ) in experiment[
        "groups"
    ].items():

        for fact_id in fact_ids:

            fact = fact_map[
                fact_id
            ]

            for prompt in fact[
                "prompts"
            ]:

                targets = [
                    (
                        "primary",
                        prompt["target"],
                    )
                ]

                targets.extend(
                    (
                        "alternative",
                        x,
                    )
                    for x in prompt.get(
                        "alternative_targets",
                        [],
                    )
                )

                targets.extend(
                    (
                        "counterfactual",
                        x,
                    )
                    for x in prompt.get(
                        "counterfactual_targets",
                        [],
                    )
                )

                for (
                    target_type,
                    target,
                ) in targets:

                    scores = (
                        score_target_sequence(
                            model,
                            tokenizer,
                            prompt["text"],
                            target,
                        )
                    )

                    rows.append({
                        "model_id":
                            "M_ref",

                        "checkpoint":
                            MREF_CHECKPOINT.name,

                        "experiment_id":
                            EXPECTED_EXPERIMENT_ID,

                        "group":
                            group_name,

                        "fact_id":
                            fact_id,

                        "subject":
                            fact["subject"],

                        "relation":
                            fact["relation"],

                        "object":
                            fact["object"],

                        "prompt_id":
                            prompt["prompt_id"],

                        "prompt_type":
                            prompt[
                                "prompt_type"
                            ],

                        "orientation":
                            prompt.get(
                                "orientation"
                            ),

                        "prompt":
                            prompt["text"],

                        "target":
                            target,

                        "target_type":
                            target_type,

                        **scores,
                    })

    return pd.DataFrame(
        rows
    )


# ================================================================
# 7. Gruppenlogik identisch zu GA
# ================================================================

def derive_analysis_group(
    row
):
    group = row["group"]

    if group not in {
        "target",
        "same_relation",
    }:
        return group

    orientation = row.get(
        "orientation"
    )

    if (
        orientation
        == "country_to_capital"
    ):
        return (
            f"{group}_direct"
        )

    if (
        orientation
        == "capital_to_country"
    ):
        return (
            f"{group}_inverse"
        )

    prompt_type = str(
        row.get(
            "prompt_type",
            "",
        )
    ).lower()

    if "inverse" in prompt_type:
        return (
            f"{group}_inverse"
        )

    if (
        "direct" in prompt_type
        or "possessive"
        in prompt_type
    ):
        return (
            f"{group}_direct"
        )

    raise ValueError(
        "Prompt-Richtung konnte "
        "nicht bestimmt werden: "
        f"{row['prompt_id']}"
    )


def aggregate_reference_results(
    results
):
    primary = results[
        results["target_type"]
        == "primary"
    ].copy()

    primary[
        "analysis_group"
    ] = primary.apply(
        derive_analysis_group,
        axis=1,
    )

    fact_summary = (
        primary
        .groupby(
            [
                "group",
                "analysis_group",
                "fact_id",
                "subject",
                "relation",
                "object",
            ],
            as_index=False,
            dropna=False,
        )
        .agg(
            mean_sequence_nll=(
                "sequence_nll",
                "mean",
            ),

            std_sequence_nll=(
                "sequence_nll",
                "std",
            ),

            mean_geo_probability=(
                "geo_mean_probability",
                "mean",
            ),

            median_first_token_rank=(
                "first_token_rank",
                "median",
            ),

            max_first_token_rank=(
                "first_token_rank",
                "max",
            ),

            prompt_count=(
                "prompt_id",
                "nunique",
            ),

            target_token_count=(
                "target_token_count",
                "max",
            ),
        )
    )

    fact_summary[
        "std_sequence_nll"
    ] = (
        fact_summary[
            "std_sequence_nll"
        ]
        .fillna(0.0)
    )

    group_summary = (
        fact_summary
        .groupby(
            "analysis_group",
            as_index=False,
        )
        .agg(
            fact_count=(
                "fact_id",
                "nunique",
            ),

            prompt_count=(
                "prompt_count",
                "sum",
            ),

            mean_fact_nll=(
                "mean_sequence_nll",
                "mean",
            ),

            std_fact_nll=(
                "mean_sequence_nll",
                "std",
            ),

            median_fact_rank=(
                "median_first_token_rank",
                "median",
            ),

            mean_fact_geo_probability=(
                "mean_geo_probability",
                "mean",
            ),
        )
    )

    group_summary[
        "std_fact_nll"
    ] = (
        group_summary[
            "std_fact_nll"
        ]
        .fillna(0.0)
    )

    return (
        primary,
        fact_summary,
        group_summary,
    )


# ================================================================
# 8. M_ref laden und evaluieren
# ================================================================

print(
    "\nLade M_ref ..."
)

MREF = load_model_from_checkpoint(
    MREF_CHECKPOINT,
    MODEL_CONFIG,
    DEVICE,
)

print(
    "M_ref geladen auf:",
    next(
        MREF.parameters()
    ).device,
)


reference_evaluation = (
    evaluate_reference_model(
        MREF,
        tokenizer,
    )
)

(
    reference_primary,
    reference_fact_summary,
    reference_group_summary,
) = aggregate_reference_results(
    reference_evaluation
)


# ================================================================
# 9. Sanity Checks
# ================================================================

print(
    "\nPrompt-Target-"
    "Evaluationen:",
    len(reference_evaluation),
)

print(
    "Primary-Prompts:",
    len(reference_primary),
)

print(
    "Fakten:",
    reference_fact_summary[
        "fact_id"
    ].nunique(),
)

print(
    "Analysegruppen:",
    sorted(
        reference_group_summary[
            "analysis_group"
        ].tolist()
    ),
)

# Finaler Benchmark hatte 55 Prompt-Target-Evaluationen.
assert (
    len(reference_evaluation)
    == 55
), (
    f"55 Evaluationen erwartet, "
    f"gefunden: "
    f"{len(reference_evaluation)}"
)

assert (
    reference_fact_summary[
        "fact_id"
    ].nunique()
    == 20
)


# ================================================================
# 10. Gegen M0 aus einem finalen Run vergleichen
# ================================================================

M0_EVAL_PATH = Path(
    "/content/drive/MyDrive/"
    "overall_results/raw/"
    "full_window/lr_1e-06/"
    "evaluation_history.csv"
)

assert M0_EVAL_PATH.exists(), (
    f"M0-Evaluation fehlt: "
    f"{M0_EVAL_PATH}"
)

m0_all = pd.read_csv(
    M0_EVAL_PATH
)

m0_eval = (
    m0_all[
        m0_all[
            "unlearning_step"
        ]
        == 0
    ]
    .copy()
)

assert len(m0_eval) == 55


JOIN_COLUMNS = [
    "experiment_id",
    "group",
    "fact_id",
    "prompt_id",
    "target_type",
    "target",
]

METRIC_COLUMNS = [
    "sequence_nll",
    "geo_mean_probability",
    "first_token_rank",
    "first_token_probability",
]


reference_vs_m0 = (
    m0_eval[
        JOIN_COLUMNS
        + METRIC_COLUMNS
    ]
    .merge(
        reference_evaluation[
            JOIN_COLUMNS
            + METRIC_COLUMNS
        ],
        on=JOIN_COLUMNS,
        how="inner",
        suffixes=(
            "_M0",
            "_Mref",
        ),
        validate="one_to_one",
    )
)

assert (
    len(reference_vs_m0)
    == 55
)

reference_vs_m0[
    "delta_sequence_nll"
] = (
    reference_vs_m0[
        "sequence_nll_Mref"
    ]
    - reference_vs_m0[
        "sequence_nll_M0"
    ]
)

reference_vs_m0[
    "delta_first_token_rank"
] = (
    reference_vs_m0[
        "first_token_rank_Mref"
    ]
    - reference_vs_m0[
        "first_token_rank_M0"
    ]
)

reference_vs_m0[
    "probability_ratio"
] = (
    reference_vs_m0[
        "geo_mean_probability_Mref"
    ]
    / reference_vs_m0[
        "geo_mean_probability_M0"
    ].clip(
        lower=1e-30
    )
)


# ================================================================
# 11. M0 auch auf Gruppenebene aggregieren
# ================================================================

m0_primary = m0_eval[
    m0_eval["target_type"]
    == "primary"
].copy()

m0_primary[
    "analysis_group"
] = m0_primary.apply(
    derive_analysis_group,
    axis=1,
)

m0_fact_summary = (
    m0_primary
    .groupby(
        [
            "group",
            "analysis_group",
            "fact_id",
        ],
        as_index=False,
    )
    .agg(
        mean_sequence_nll=(
            "sequence_nll",
            "mean",
        )
    )
)

m0_group_summary = (
    m0_fact_summary
    .groupby(
        "analysis_group",
        as_index=False,
    )
    .agg(
        m0_mean_fact_nll=(
            "mean_sequence_nll",
            "mean",
        )
    )
)


reference_group_comparison = (
    reference_group_summary
    .merge(
        m0_group_summary,
        on="analysis_group",
        how="left",
        validate="one_to_one",
    )
)

reference_group_comparison[
    "delta_mean_fact_nll_vs_m0"
] = (
    reference_group_comparison[
        "mean_fact_nll"
    ]
    - reference_group_comparison[
        "m0_mean_fact_nll"
    ]
)


# ================================================================
# 12. Speichern
# ================================================================

reference_evaluation.to_csv(
    OUTPUT_DIR
    / "reference_evaluation.csv",
    index=False,
)

reference_primary.to_csv(
    OUTPUT_DIR
    / "reference_primary_prompts.csv",
    index=False,
)

reference_fact_summary.to_csv(
    OUTPUT_DIR
    / "reference_fact_summary.csv",
    index=False,
)

reference_group_summary.to_csv(
    OUTPUT_DIR
    / "reference_group_summary.csv",
    index=False,
)

reference_vs_m0.to_csv(
    OUTPUT_DIR
    / "reference_vs_m0_prompt_comparison.csv",
    index=False,
)

reference_group_comparison.to_csv(
    OUTPUT_DIR
    / "reference_vs_m0_group_comparison.csv",
    index=False,
)


# ================================================================
# 13. Wichtigste Ergebnisse anzeigen
# ================================================================

print(
    "\nM_ref – Gruppenvergleich "
    "gegen M0:"
)

display(
    reference_group_comparison[
        [
            "analysis_group",
            "fact_count",
            "prompt_count",
            "m0_mean_fact_nll",
            "mean_fact_nll",
            "delta_mean_fact_nll_vs_m0",
            "median_fact_rank",
            "mean_fact_geo_probability",
        ]
    ]
)


print(
    "\nM_ref – Target-Prompts:"
)

display(
    reference_primary[
        reference_primary[
            "group"
        ]
        == "target"
    ][
        [
            "prompt_id",
            "orientation",
            "prompt",
            "target",
            "sequence_nll",
            "geo_mean_probability",
            "first_token_probability",
            "first_token_rank",
        ]
    ]
)


print("\nGespeichert unter:")
for filename in [
    "reference_evaluation.csv",
    "reference_primary_prompts.csv",
    "reference_fact_summary.csv",
    "reference_group_summary.csv",
    "reference_vs_m0_prompt_comparison.csv",
    "reference_vs_m0_group_comparison.csv",
]:
    print(
        " -",
        OUTPUT_DIR / filename,
    )


# Speicher freigeben
del MREF
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ================================================================
# Pfade
# ================================================================

ROOT = Path(
    "/content/drive/MyDrive/overall_results"
)

TABLES = ROOT / "outputs" / "tables"
FIGURES = ROOT / "outputs" / "figures"

FIGURES.mkdir(
    parents=True,
    exist_ok=True,
)


# ================================================================
# Eingabedateien
# ================================================================

SAME_LR_PATH = (
    TABLES / "same_lr_step16_comparison.csv"
)

TOKEN_SELECTIVITY_PATH = (
    TABLES / "token_selectivity_step16.csv"
)

MATCHED_PATH = (
    TABLES / "matched_forgetting_top20_target.csv"
)

FULL_VALIDATION_PATH = (
    TABLES / "full_validation_results.csv"
)


for path in [
    SAME_LR_PATH,
    TOKEN_SELECTIVITY_PATH,
    MATCHED_PATH,
    FULL_VALIDATION_PATH,
]:
    assert path.exists(), (
        f"Datei fehlt: {path}"
    )


# ================================================================
# Tabellen laden
# ================================================================

same_lr = pd.read_csv(
    SAME_LR_PATH
)

token_selectivity = pd.read_csv(
    TOKEN_SELECTIVITY_PATH
)

matched = pd.read_csv(
    MATCHED_PATH
)

full_validation = pd.read_csv(
    FULL_VALIDATION_PATH
)


# ================================================================
# Einheitliche Darstellung
# ================================================================

plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 120,
    "savefig.dpi": 300,
})


LR_LABELS = {
    1e-6: r"$1\times10^{-6}$",
    5e-6: r"$5\times10^{-6}$",
    1e-5: r"$1\times10^{-5}$",
    1.5e-5: r"$1{,}5\times10^{-5}$",
    2e-5: r"$2\times10^{-5}$",
    3e-5: r"$3\times10^{-5}$",
}


print("Figuren werden gespeichert unter:")
print(FIGURES)

In [ ]:
# ================================================================
# Abbildung 1:
# Target-Delta-NLL über Lernrate
# ================================================================

df = (
    same_lr
    .sort_values("analysis_learning_rate")
    .reset_index(drop=True)
)

x = np.arange(
    len(df)
)

fig, ax = plt.subplots(
    figsize=(7.2, 4.5)
)

ax.plot(
    x,
    df["full_target_all_delta_nll"],
    marker="o",
    linewidth=2,
    label="Full-Sequence GA",
)

ax.plot(
    x,
    df["masked_target_all_delta_nll"],
    marker="s",
    linewidth=2,
    label="Token-selektives GA",
)

ax.set_xticks(x)

ax.set_xticklabels([
    LR_LABELS[float(lr)]
    for lr in df[
        "analysis_learning_rate"
    ]
])

ax.set_xlabel(
    "Lernrate"
)

ax.set_ylabel(
    r"Mittlere Target-$\Delta$NLL"
)

ax.axhline(
    0,
    linewidth=0.8,
)

ax.grid(
    axis="y",
    alpha=0.25,
)

ax.legend()

fig.tight_layout()

png_path = (
    FIGURES
    / "target_delta_nll_lr.png"
)

pdf_path = (
    FIGURES
    / "target_delta_nll_lr.pdf"
)

fig.savefig(
    png_path,
    bbox_inches="tight",
)

fig.savefig(
    pdf_path,
    bbox_inches="tight",
)

plt.show()

print("Gespeichert:", png_path)

In [ ]:
# ================================================================
# Abbildung 2:
# Forget-Utility-Trade-off
# ================================================================

# Target-Effekt aus Same-LR-Tabelle in Long-Format
target_full = same_lr[
    [
        "analysis_learning_rate",
        "full_target_all_delta_nll",
    ]
].copy()

target_full["method"] = "full_window"

target_full = target_full.rename(
    columns={
        "full_target_all_delta_nll":
            "target_delta_nll"
    }
)


target_masked = same_lr[
    [
        "analysis_learning_rate",
        "masked_target_all_delta_nll",
    ]
].copy()

target_masked["method"] = "masked"

target_masked = target_masked.rename(
    columns={
        "masked_target_all_delta_nll":
            "target_delta_nll"
    }
)


target_long = pd.concat(
    [
        target_full,
        target_masked,
    ],
    ignore_index=True,
)


# Full Validation Step 16
utility = full_validation[
    full_validation["method"].isin(
        [
            "full_window",
            "masked",
        ]
    )
].copy()

utility = utility.rename(
    columns={
        "learning_rate":
            "analysis_learning_rate"
    }
)


tradeoff = target_long.merge(
    utility[
        [
            "method",
            "analysis_learning_rate",
            "delta_full_validation_loss",
        ]
    ],
    on=[
        "method",
        "analysis_learning_rate",
    ],
    how="inner",
    validate="one_to_one",
)

assert len(tradeoff) == 12


fig, ax = plt.subplots(
    figsize=(7.2, 4.8)
)


for method, label, marker in [
    (
        "full_window",
        "Full-Sequence GA",
        "o",
    ),
    (
        "masked",
        "Token-selektives GA",
        "s",
    ),
]:
    subset = (
        tradeoff[
            tradeoff["method"]
            == method
        ]
        .sort_values(
            "analysis_learning_rate"
        )
    )

    ax.plot(
        subset["target_delta_nll"],
        subset[
            "delta_full_validation_loss"
        ],
        marker=marker,
        linewidth=1.8,
        label=label,
    )

  # Nur ausgewählte Lernraten beschriften
ANNOTATE_LRS = {
    1e-5,
    1.5e-5,
    2e-5,
    3e-5,
}

for _, row in subset.iterrows():

    lr = float(
        row["analysis_learning_rate"]
    )

    if any(
        np.isclose(lr, x)
        for x in ANNOTATE_LRS
    ):
        ax.annotate(
            LR_LABELS[lr],
            (
                row["target_delta_nll"],
                row["delta_full_validation_loss"],
            ),
            xytext=(6, 5),
            textcoords="offset points",
            fontsize=8,
        )


ax.axhline(
    0,
    color="black",
    linestyle="--",
    linewidth=0.8,
    alpha=0.5,
)

ax.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=0.8,
    alpha=0.5,
)

ax.set_xlabel(
    r"Mittlere Target-$\Delta$NLL"
)

ax.set_ylabel(
    r"$\Delta L_{\mathrm{val}}$ "
    r"(vollständige Validation)"
)

ax.grid(
    alpha=0.25,
)

ax.legend()

fig.tight_layout()

png_path = (
    FIGURES
    / "forget_utility_tradeoff.png"
)

pdf_path = (
    FIGURES
    / "forget_utility_tradeoff.pdf"
)

fig.savefig(
    png_path,
    bbox_inches="tight",
)

fig.savefig(
    pdf_path,
    bbox_inches="tight",
)

plt.show()

print("Gespeichert:", png_path)

In [ ]:
# ================================================================
# Abbildung 3:
# Tokenbezogene Selektivität
# ================================================================

df = token_selectivity.copy()


fig, ax = plt.subplots(
    figsize=(7.0, 5.2)
)


for method, label, marker in [
    (
        "full_window",
        "Full-Sequence GA",
        "o",
    ),
    (
        "masked",
        "Token-selektives GA",
        "s",
    ),
]:
    subset = (
        df[
            df["analysis_method"]
            == method
        ]
        .sort_values(
            "analysis_learning_rate"
        )
    )

    ax.plot(
        subset["context_delta_nll"],
        subset["relation_delta_nll"],
        marker=marker,
        linewidth=1.8,
        label=label,
    )

ANNOTATE_LRS = {
    1e-5,
    2e-5,
    3e-5,
}

for _, row in subset.iterrows():

    lr = float(
        row["analysis_learning_rate"]
    )

    if any(
        np.isclose(lr, x)
        for x in ANNOTATE_LRS
    ):
        ax.annotate(
            LR_LABELS[lr],
            (
                row["context_delta_nll"],
                row["relation_delta_nll"],
            ),
            xytext=(6, 4),
            textcoords="offset points",
            fontsize=8,
        )

ax.set_xlabel(
    r"$\Delta L_{\mathrm{ctx}}$ "
    "(übrige 16\u202f368 Positionen)"
)

ax.set_ylabel(
    r"$\Delta L_{\mathrm{rel}}$ "
    "(16 relationstragende Positionen)"
)

ax.axhline(
    0,
    color="black",
    linestyle="--",
    linewidth=0.8,
    alpha=0.5,
)

ax.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=0.8,
    alpha=0.5,
)

ax.grid(
    alpha=0.25,
)

ax.legend()

fig.tight_layout()

png_path = (
    FIGURES
    / "token_selectivity.png"
)

pdf_path = (
    FIGURES
    / "token_selectivity.pdf"
)

fig.savefig(
    png_path,
    bbox_inches="tight",
)

fig.savefig(
    pdf_path,
    bbox_inches="tight",
)

plt.show()

print("Gespeichert:", png_path)

In [ ]:
# ================================================================
# Abbildung 4:
# Kollateraleffekte beim Matched-Forgetting
# ================================================================

MATCH_FULL_LR = 2e-5
MATCH_FULL_STEP = 5

MATCH_MASKED_LR = 1e-6
MATCH_MASKED_STEP = 10


match_row = matched[
    np.isclose(
        matched[
            "analysis_learning_rate_full"
        ],
        MATCH_FULL_LR,
    )
    &
    (
        matched[
            "unlearning_step_full"
        ]
        == MATCH_FULL_STEP
    )
    &
    np.isclose(
        matched[
            "analysis_learning_rate_masked"
        ],
        MATCH_MASKED_LR,
    )
    &
    (
        matched[
            "unlearning_step_masked"
        ]
        == MATCH_MASKED_STEP
    )
]


assert len(match_row) == 1, (
    "Das festgelegte Matched-Paar "
    "wurde nicht eindeutig gefunden."
)

row = match_row.iloc[0]


groups = [
    "same_subject",
    "same_object",
    "same_relation_direct",
    "same_relation_inverse",
    "same_window",
    "unrelated",
]

labels = [
    "same_subject",
    "same_object",
    "same_relation\ndirekt",
    "same_relation\ninvers",
    "same_window",
    "unrelated",
]


full_values = np.array([
    row[f"{group}_full"]
    for group in groups
])

masked_values = np.array([
    row[f"{group}_masked"]
    for group in groups
])


y = np.arange(
    len(groups)
)

height = 0.36


fig, ax = plt.subplots(
    figsize=(7.4, 4.8)
)


ax.barh(
    y - height / 2,
    full_values,
    height=height,
    label="Full-Sequence GA",
)

ax.barh(
    y + height / 2,
    masked_values,
    height=height,
    label="Token-selektives GA",
)


ax.set_yticks(y)

ax.set_yticklabels(
    labels
)

ax.invert_yaxis()

ax.axvline(
    0,
    linewidth=0.8,
)

ax.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=0.8,
    alpha=0.6,
)

ax.grid(
    axis="x",
    alpha=0.25,
)

ax.legend()

fig.tight_layout()

png_path = (
    FIGURES
    / "matched_locality.png"
)

pdf_path = (
    FIGURES
    / "matched_locality.pdf"
)

fig.savefig(
    png_path,
    bbox_inches="tight",
)

fig.savefig(
    pdf_path,
    bbox_inches="tight",
)

plt.show()

print("Gespeichert:", png_path)


In [ ]:
expected_figures = [
    "target_delta_nll_lr.png",
    "forget_utility_tradeoff.png",
    "token_selectivity.png",
    "matched_locality.png",
]

print("Erzeugte Abbildungen:\n")

for filename in expected_figures:
    path = FIGURES / filename

    print(
        "OK" if path.exists() else "FEHLT",
        path,
    )

In [ ]:
# ================================================================
# Counterfactual-Auswertung der Target-Prompts
# ================================================================

target_eval = evaluation_all[
    evaluation_all["group"] == "target"
].copy()

# ---------------------------------------------------------------
# 1. Step-0-Werte innerhalb jedes Runs als Baseline
# ---------------------------------------------------------------

baseline_keys = [
    "analysis_method",
    "analysis_learning_rate",
    "prompt_id",
    "target_type",
    "target",
]

target_baseline = (
    target_eval[
        target_eval["unlearning_step"] == 0
    ][baseline_keys + ["sequence_nll"]]
    .rename(
        columns={
            "sequence_nll": "baseline_sequence_nll"
        }
    )
)

target_eval_delta = target_eval.merge(
    target_baseline,
    on=baseline_keys,
    how="left",
    validate="many_to_one",
)

target_eval_delta["delta_nll"] = (
    target_eval_delta["sequence_nll"]
    - target_eval_delta["baseline_sequence_nll"]
)

assert target_eval_delta["baseline_sequence_nll"].notna().all()


# ---------------------------------------------------------------
# 2. Primäre Zielantwort pro Prompt
# ---------------------------------------------------------------

primary_delta = target_eval_delta[
    target_eval_delta["target_type"] == "primary"
][
    [
        "analysis_method",
        "analysis_learning_rate",
        "unlearning_step",
        "prompt_id",
        "orientation",
        "delta_nll",
    ]
].rename(
    columns={
        "delta_nll": "target_delta_nll"
    }
)


# ---------------------------------------------------------------
# 3. Counterfactuals pro Prompt aggregieren
# ---------------------------------------------------------------

counterfactual_delta = (
    target_eval_delta[
        target_eval_delta["target_type"] == "counterfactual"
    ]
    .groupby(
        [
            "analysis_method",
            "analysis_learning_rate",
            "unlearning_step",
            "prompt_id",
            "orientation",
        ],
        as_index=False,
    )
    .agg(
        counterfactual_count=("target", "nunique"),
        counterfactual_mean_delta_nll=("delta_nll", "mean"),
        counterfactual_min_delta_nll=("delta_nll", "min"),
        counterfactual_max_delta_nll=("delta_nll", "max"),
    )
)

# Im finalen Benchmark existieren drei Counterfactuals pro Target-Prompt.
assert (
    counterfactual_delta["counterfactual_count"] == 3
).all()


# ---------------------------------------------------------------
# 4. Deskriptive Differenz zwischen Target und Counterfactuals
# ---------------------------------------------------------------

counterfactual_prompt_history = primary_delta.merge(
    counterfactual_delta,
    on=[
        "analysis_method",
        "analysis_learning_rate",
        "unlearning_step",
        "prompt_id",
        "orientation",
    ],
    how="inner",
    validate="one_to_one",
)

counterfactual_prompt_history["specificity_score"] = (
    counterfactual_prompt_history["target_delta_nll"]
    - counterfactual_prompt_history[
        "counterfactual_mean_delta_nll"
    ]
)

# 5 Target-Prompts pro Modellzustand
prompt_counts = (
    counterfactual_prompt_history
    .groupby(
        [
            "analysis_method",
            "analysis_learning_rate",
            "unlearning_step",
        ]
    )["prompt_id"]
    .nunique()
)

assert (prompt_counts == 5).all()


# ---------------------------------------------------------------
# 5. Über alle fünf Prompts aggregieren
# ---------------------------------------------------------------

counterfactual_summary = (
    counterfactual_prompt_history
    .groupby(
        [
            "analysis_method",
            "analysis_learning_rate",
            "unlearning_step",
        ],
        as_index=False,
    )
    .agg(
        target_delta_nll=("target_delta_nll", "mean"),
        counterfactual_delta_nll=(
            "counterfactual_mean_delta_nll",
            "mean",
        ),
        specificity_score=("specificity_score", "mean"),
        prompt_count=("prompt_id", "nunique"),
    )
)

assert len(counterfactual_summary) == 72


# ---------------------------------------------------------------
# 6. Zusätzlich direkte / inverse Richtung
# ---------------------------------------------------------------

counterfactual_by_direction = (
    counterfactual_prompt_history
    .groupby(
        [
            "analysis_method",
            "analysis_learning_rate",
            "unlearning_step",
            "orientation",
        ],
        as_index=False,
    )
    .agg(
        target_delta_nll=("target_delta_nll", "mean"),
        counterfactual_delta_nll=(
            "counterfactual_mean_delta_nll",
            "mean",
        ),
        specificity_score=("specificity_score", "mean"),
        prompt_count=("prompt_id", "nunique"),
    )
)


# ---------------------------------------------------------------
# 7. Step-16-Auswertung
# ---------------------------------------------------------------

counterfactual_step16 = (
    counterfactual_summary[
        counterfactual_summary["unlearning_step"] == 16
    ]
    .sort_values(
        [
            "analysis_method",
            "analysis_learning_rate",
        ]
    )
    .reset_index(drop=True)
)

assert len(counterfactual_step16) == 12

print("Counterfactual-Auswertung nach 16 Updates:")
display(counterfactual_step16)


# ---------------------------------------------------------------
# 8. Matched-Forgetting-Paar
# ---------------------------------------------------------------

matched_counterfactual = counterfactual_summary[
    (
        (counterfactual_summary["analysis_method"] == "full_window")
        & np.isclose(
            counterfactual_summary["analysis_learning_rate"],
            2e-5,
        )
        & (counterfactual_summary["unlearning_step"] == 5)
    )
    |
    (
        (counterfactual_summary["analysis_method"] == "masked")
        & np.isclose(
            counterfactual_summary["analysis_learning_rate"],
            1e-6,
        )
        & (counterfactual_summary["unlearning_step"] == 10)
    )
].copy()

assert len(matched_counterfactual) == 2

print("\nCounterfactual-Auswertung des Matched-Forgetting-Paars:")
display(matched_counterfactual)


# ---------------------------------------------------------------
# 9. Export
# ---------------------------------------------------------------

counterfactual_prompt_history.to_csv(
    TABLES / "counterfactual_prompt_history.csv",
    index=False,
)

counterfactual_summary.to_csv(
    TABLES / "counterfactual_specificity_history.csv",
    index=False,
)

counterfactual_by_direction.to_csv(
    TABLES / "counterfactual_specificity_by_direction.csv",
    index=False,
)

counterfactual_step16.to_csv(
    TABLES / "counterfactual_specificity_step16.csv",
    index=False,
)

matched_counterfactual.to_csv(
    TABLES / "counterfactual_specificity_matched.csv",
    index=False,
)

print("\nCounterfactual-Tabellen gespeichert.")
